# 02_04 — Inventario integrado de aparcamientos EMT y municipales

## Objetivo

Construir un inventario estático, reproducible y georreferenciado de aparcamientos off-street de Madrid que permita:

- representar los aparcamientos en el mapa integrado del TFM;
- identificar alternativas al estacionamiento regulado SER;
- distinguir aparcamientos utilizables por público general;
- conservar las claves necesarias para enlazar posteriormente ocupación histórica, mensual y en tiempo real;
- evitar duplicidades entre las fuentes EMT y municipal.

La unidad espacial es el **aparcamiento** y la granularidad temporal es **estática**.

Este notebook no modela ocupación ni genera recomendaciones. Su responsabilidad termina en la construcción y validación del inventario integrado.

## 1. Contrato metodológico

### Fuentes

1. `emt_parkings`: inventario principal y fuente ancla para los joins posteriores con ocupación EMT.
2. `emt_aparcamientos_publicos`: fuente municipal utilizada para enriquecer, clasificar y ampliar el inventario.

### Principios

- Cada fuente se inspecciona y limpia individualmente antes de integrarla.
- Los identificadores originales se conservan para futuros joins.
- Los nombres se normalizan sin eliminar el valor original.
- Las coordenadas se validan antes del matching.
- Los registros no se fusionan únicamente por proximidad.
- Los aparcamientos de residentes se conservan en el inventario, pero no se consideran recomendables para público general.
- Los campos auxiliares de diagnóstico no se conservan en los outputs finales salvo necesidad explícita.

## 2. Configuración y rutas

In [20]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)

# Localización reproducible de la raíz del repositorio.
current_path = Path.cwd().resolve()
root_candidates = [current_path, *current_path.parents]

ROOT = next(
    (
        candidate
        for candidate in root_candidates
        if (candidate / "data_catalog.csv").exists()
        and (candidate / "data").exists()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError(
        "No se ha podido localizar la raíz de TFM_parking_madrid."
    )

EMT_RAW_PATH = (
    ROOT
    / "data/raw/emt/emt_parkings"
    / "emt_parkings__actual.csv"
)

EMT_INTERIM_PATH = (
    ROOT
    / "data/interim/emt/emt_parkings"
    / "emt_parkings_clean.parquet"
)

# Escritura activada tras la validación final del notebook.
WRITE_OUTPUTS = True


def clean_visible_text(value):
    """Limpia espacios y saltos de línea sin alterar el contenido nominal."""
    if pd.isna(value):
        return pd.NA

    value = str(value).replace("\r", " ").replace("\n", " ")
    value = re.sub(r"\s+", " ", value).strip()

    return value if value else pd.NA


def normalize_match_text(value):
    """Genera texto auxiliar para detectar coincidencias y duplicados."""
    if pd.isna(value):
        return ""

    value = unicodedata.normalize("NFKD", str(value))
    value = "".join(
        character
        for character in value
        if not unicodedata.combining(character)
    )
    value = value.lower().strip()
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


print("ROOT:", ROOT)
print("Input EMT:", EMT_RAW_PATH)
print("Output EMT:", EMT_INTERIM_PATH)
print("WRITE_OUTPUTS:", WRITE_OUTPUTS)

assert EMT_RAW_PATH.exists(), f"No existe el raw EMT: {EMT_RAW_PATH}"

MUNICIPAL_RAW_PATH = (
    ROOT
    / "data/raw/emt/emt_aparcamientos_publicos"
    / "emt_aparcamientos_publicos__actual.csv"
)

MUNICIPAL_INTERIM_PATH = (
    ROOT
    / "data/interim/emt/emt_aparcamientos_publicos"
    / "emt_aparcamientos_publicos_clean.parquet"
)


def parse_count_token(value):
    """Convierte cifras con separadores de miles en enteros."""
    if value is None or pd.isna(value):
        return pd.NA

    digits = re.sub(r"[^\d]", "", str(value))
    return int(digits) if digits else pd.NA


def extract_first_count(text, patterns):
    """Devuelve la primera cifra reconocida por una lista de patrones."""
    if pd.isna(text):
        return pd.NA

    text = clean_visible_text(text)

    for pattern in patterns:
        match = re.search(
            pattern,
            str(text),
            flags=re.IGNORECASE,
        )
        if match:
            return parse_count_token(match.group(1))

    return pd.NA


def clean_parking_name(value):
    """Elimina únicamente prefijos genéricos del nombre municipal."""
    value = clean_visible_text(value)

    if pd.isna(value):
        return pd.NA

    value = re.sub(
        r"^aparcamiento\s+"
        r"(?:p[úu]blico|mixto|disuasorio)"
        r"\.?\s*",
        "",
        str(value),
        flags=re.IGNORECASE,
    )

    value = value.strip(" .")
    return value if value else pd.NA


def spanish_title(value):
    """Normaliza mayúsculas de una dirección sin alterar conectores."""
    value = clean_visible_text(value)

    if pd.isna(value):
        return pd.NA

    lower_words = {
        "de", "del", "la", "las", "los",
        "y", "el", "en", "a",
    }

    tokens = re.split(r"(\s+|-)", str(value).lower())
    result = []
    first_word = True

    for token in tokens:
        if not token or token.isspace() or token == "-":
            result.append(token)
            continue

        if first_word or token not in lower_words:
            result.append(token.capitalize())
        else:
            result.append(token)

        first_word = False

    return "".join(result)


def format_street_number(value):
    if pd.isna(value):
        return None

    numeric = float(value)

    if numeric.is_integer():
        return str(int(numeric))

    return str(value).strip()


def build_municipal_address(row):
    street_class = spanish_title(row["CLASE-VIAL"])
    street_name = spanish_title(row["NOMBRE-VIA"])
    number = format_street_number(row["NUM"])

    street_parts = [
        str(part)
        for part in [street_class, street_name]
        if part is not None and not pd.isna(part)
    ]

    address = " ".join(street_parts)

    if number:
        address = f"{address}, {number}"

    return address.strip() if address.strip() else pd.NA


def classify_municipal_access(name, source_type):
    name_normalized = normalize_match_text(name)
    source_normalized = normalize_match_text(source_type)

    if "disuasorio" in name_normalized:
        return "disuasorio"

    if "mixto" in name_normalized:
        return "mixto"

    if "publico" in name_normalized:
        return "publico"

    if "residentes" in source_normalized:
        return "residentes"

    if "publicos" in source_normalized:
        return "publico"

    return pd.NA


assert MUNICIPAL_RAW_PATH.exists(), (
    f"No existe el raw municipal: {MUNICIPAL_RAW_PATH}"
)

print("Input municipal:", MUNICIPAL_RAW_PATH)
print("Output municipal:", MUNICIPAL_INTERIM_PATH)


ROOT: /Users/hugo/TFM_parking_madrid
Input EMT: /Users/hugo/TFM_parking_madrid/data/raw/emt/emt_parkings/emt_parkings__actual.csv
Output EMT: /Users/hugo/TFM_parking_madrid/data/interim/emt/emt_parkings/emt_parkings_clean.parquet
WRITE_OUTPUTS: True
Input municipal: /Users/hugo/TFM_parking_madrid/data/raw/emt/emt_aparcamientos_publicos/emt_aparcamientos_publicos__actual.csv
Output municipal: /Users/hugo/TFM_parking_madrid/data/interim/emt/emt_aparcamientos_publicos/emt_aparcamientos_publicos_clean.parquet


## 3. Fuente `emt_parkings`

### 3.1. Qué mide y para qué se utiliza

La fuente contiene un inventario estático de aparcamientos asociados al ecosistema de información de EMT. Cada registro representa una entrada del inventario e incluye un identificador, nombre, dirección, coordenadas y variables de capacidad.

En este TFM se utiliza como **fuente ancla del inventario integrado** porque el campo `id` permitirá enlazar posteriormente los aparcamientos con las fuentes de ocupación histórica y, cuando exista cobertura, con la información en tiempo real.

### 3.2. Columnas conservadas y descartadas

Se conservan:

- `id` → `id_emt`: clave para futuros joins con ocupación;
- `name` → `nombre_original` y `nombre`;
- `address` → `direccion`;
- `lat` y `long` → `latitud` y `longitud`;
- `isEmtParking` → `es_parking_emt`;
- `Plazas_standard` → `plazas_standard`;
- `Plazas_PMR` → `plazas_pmr`. (Plazas de Personas con Movilidad Reducida)

Se descartan:

- `areaCode`, por su elevada ausencia y falta de uso analítico;
- `email`, porque no interviene en mapas, ocupación ni joins;
- `URLIcono`, porque es un recurso visual externo;
- `state` y `town`, porque únicamente describen el ámbito territorial ya conocido y sus diferencias son de formato.

### 3.3. Comprobaciones previas

Las comprobaciones se limitan a condiciones que pueden modificar una decisión de limpieza:

- estructura esperada del fichero;
- cobertura y unicidad del identificador;
- nulos en campos esenciales;
- categorías no reconocidas;
- coordenadas fuera de un rango geográfico plausible;
- duplicados exactos de fila;
- registros con el mismo contenido excluyendo únicamente `id`;
- coordenadas exactamente repetidas, sin redondeo;
- capacidades negativas o estándar iguales a cero.

Los nombres repetidos no se consideran por sí solos evidencia de duplicidad.

In [21]:
emt_raw = pd.read_csv(
    EMT_RAW_PATH,
    encoding="utf-8-sig",
    sep=",",
)

expected_columns = [
    "id",
    "name",
    "address",
    "areaCode",
    "email",
    "URLIcono",
    "long",
    "lat",
    "state",
    "town",
    "isEmtParking",
    "Plazas_standard",
    "Plazas_PMR",
]

missing_columns = sorted(
    set(expected_columns) - set(emt_raw.columns)
)

unexpected_columns = sorted(
    set(emt_raw.columns) - set(expected_columns)
)

latitud_numeric = pd.to_numeric(
    emt_raw["lat"],
    errors="coerce",
)

longitud_numeric = pd.to_numeric(
    emt_raw["long"],
    errors="coerce",
)

invalid_is_emt_values = sorted(
    set(
        emt_raw["isEmtParking"]
        .dropna()
        .astype(str)
        .str.strip()
        .str.upper()
    )
    - {"Y", "N"}
)

content_columns_without_id = [
    column
    for column in emt_raw.columns
    if column != "id"
]

content_duplicate_mask = emt_raw.duplicated(
    subset=content_columns_without_id,
    keep=False,
)

coordinate_duplicate_mask = emt_raw.duplicated(
    subset=["lat", "long"],
    keep=False,
)

duplicate_content_records = (
    emt_raw.loc[
        content_duplicate_mask,
        ["id", "name"],
    ]
    .apply(tuple, axis=1)
    .tolist()
)

duplicate_coordinate_records = (
    emt_raw.loc[
        coordinate_duplicate_mask,
        ["id", "name", "lat", "long"],
    ]
    .apply(tuple, axis=1)
    .tolist()
)

missing_address_records = (
    emt_raw.loc[
        emt_raw["address"].isna(),
        ["id", "name"],
    ]
    .apply(tuple, axis=1)
    .tolist()
)

zero_standard_capacity_records = (
    emt_raw.loc[
        emt_raw["Plazas_standard"].eq(0),
        ["id", "name"],
    ]
    .apply(tuple, axis=1)
    .tolist()
)

emt_checks = pd.DataFrame(
    [
        ("n_registros", len(emt_raw)),
        ("n_columnas", len(emt_raw.columns)),
        ("columnas_esperadas_ausentes", missing_columns),
        ("columnas_inesperadas", unexpected_columns),
        ("id_nulos", int(emt_raw["id"].isna().sum())),
        (
            "id_duplicados_extra",
            int(
                emt_raw["id"]
                .duplicated(keep="first")
                .sum()
            ),
        ),
        ("nombre_nulos", int(emt_raw["name"].isna().sum())),
        (
            "direccion_nulos",
            int(emt_raw["address"].isna().sum()),
        ),
        (
            "registros_direccion_nula",
            missing_address_records,
        ),
        ("latitud_nulos", int(latitud_numeric.isna().sum())),
        ("longitud_nulos", int(longitud_numeric.isna().sum())),
        (
            "coordenadas_fuera_rango_amplio",
            int(
                (
                    ~latitud_numeric.between(40.0, 41.0)
                    | ~longitud_numeric.between(-4.5, -3.0)
                ).sum()
            ),
        ),
        (
            "filas_duplicadas_exactas_extra",
            int(
                emt_raw
                .duplicated(keep="first")
                .sum()
            ),
        ),
        (
            "registros_duplicados_sin_id_extra",
            int(
                emt_raw
                .duplicated(
                    subset=content_columns_without_id,
                    keep="first",
                )
                .sum()
            ),
        ),
        (
            "registros_duplicados_sin_id",
            duplicate_content_records,
        ),
        (
            "grupos_coordenadas_exactas_repetidas",
            int(
                emt_raw.loc[
                    coordinate_duplicate_mask,
                    ["lat", "long"],
                ]
                .drop_duplicates()
                .shape[0]
            ),
        ),
        (
            "registros_coordenadas_exactas_repetidas",
            duplicate_coordinate_records,
        ),
        (
            "isEmtParking_valores_no_reconocidos",
            invalid_is_emt_values,
        ),
        (
            "plazas_standard_negativas",
            int(emt_raw["Plazas_standard"].lt(0).sum()),
        ),
        (
            "plazas_pmr_negativas",
            int(emt_raw["Plazas_PMR"].lt(0).sum()),
        ),
        (
            "plazas_standard_igual_cero",
            int(emt_raw["Plazas_standard"].eq(0).sum()),
        ),
        (
            "registros_plazas_standard_igual_cero",
            zero_standard_capacity_records,
        ),
    ],
    columns=["check", "valor"],
)

display(emt_checks)

,check,valor
0,n_registros,87
1,n_columnas,13
2,columnas_esperadas_ausentes,[]
3,columnas_inesperadas,[]
4,id_nulos,0
5,id_duplicados_extra,0
6,nombre_nulos,0
7,direccion_nulos,2
8,registros_direccion_nula,"[(104, Velázquez-JuanBravo), (105, Arquitecto Ribera)]"
9,latitud_nulos,0


### 3.4. Lectura de las comprobaciones y decisiones

El fichero contiene 87 registros y las 13 columnas esperadas. No se detectan columnas ausentes o adicionales, identificadores nulos o repetidos, nombres nulos, coordenadas ausentes, coordenadas fuera del rango geográfico amplio utilizado ni capacidades negativas.

Tampoco aparecen filas completamente duplicadas ni registros con el mismo contenido en todas las variables salvo el identificador. Por tanto, la limpieza individual no requiere eliminar registros duplicados.

Se identifican dos direcciones ausentes, correspondientes a los aparcamientos con identificadores 104 y 105. Ambos registros comparten además exactamente la misma pareja de coordenadas, aunque presentan nombres diferentes. No se consideran duplicados ni se eliminan, ya que sus identificadores pueden ser necesarios para futuros joins. La incidencia se resolverá durante la integración con la fuente municipal antes de representarlos como dos puntos cartográficos independientes.

Cinco registros presentan Plazas_standard = 0. Como este valor no representa una capacidad operativa útil y no puede distinguirse de una capacidad no informada, se transforma en NA. Si la fuente municipal proporciona una capacidad válida para alguno de estos aparcamientos, podrá incorporarse posteriormente tras validar la correspondencia entre registros. Los ceros de `Plazas_PMR` se conservan porque sí pueden representar que el aparcamiento no dispone de plazas PMR registradas.

El indicador `isEmtParking` solo contiene los valores previstos `Y` y `N`, por lo que se transforma a una variable booleana. Las columnas `state` y `town` no se conservan porque no aportan información necesaria para el mapa, los joins ni el análisis de ocupación.

### 3.5. Limpieza

In [22]:
emt_clean = (
    emt_raw[
        [
            "id",
            "name",
            "address",
            "lat",
            "long",
            "isEmtParking",
            "Plazas_standard",
            "Plazas_PMR",
        ]
    ]
    .rename(
        columns={
            "id": "id_emt",
            "name": "nombre_original",
            "address": "direccion",
            "lat": "latitud",
            "long": "longitud",
            "isEmtParking": "es_parking_emt",
            "Plazas_standard": "plazas_standard",
            "Plazas_PMR": "plazas_pmr",
        }
    )
    .copy()
)

emt_clean["id_emt"] = pd.to_numeric(
    emt_clean["id_emt"],
    errors="raise",
).astype("Int64")

emt_clean["nombre_original"] = (
    emt_clean["nombre_original"]
    .astype("string")
)

emt_clean.insert(
    2,
    "nombre",
    emt_clean["nombre_original"]
    .map(clean_visible_text)
    .astype("string"),
)

emt_clean["direccion"] = (
    emt_clean["direccion"]
    .map(clean_visible_text)
    .astype("string")
)

emt_clean["latitud"] = pd.to_numeric(
    emt_clean["latitud"],
    errors="raise",
).astype("float64")

emt_clean["longitud"] = pd.to_numeric(
    emt_clean["longitud"],
    errors="raise",
).astype("float64")

emt_clean["es_parking_emt"] = (
    emt_clean["es_parking_emt"]
    .astype("string")
    .str.strip()
    .str.upper()
    .map({"Y": True, "N": False})
    .astype("boolean")
)

emt_clean["plazas_standard"] = (
    pd.to_numeric(
        emt_clean["plazas_standard"],
        errors="raise",
    )
    .astype("Int64")
    .mask(lambda series: series.eq(0), pd.NA)
)

emt_clean["plazas_pmr"] = pd.to_numeric(
    emt_clean["plazas_pmr"],
    errors="raise",
).astype("Int64")

final_emt_columns = [
    "id_emt",
    "nombre_original",
    "nombre",
    "direccion",
    "latitud",
    "longitud",
    "es_parking_emt",
    "plazas_standard",
    "plazas_pmr",
]

emt_clean = emt_clean[final_emt_columns]

# Comprobaciones silenciosas del contrato.
assert len(emt_clean) == len(emt_raw)
assert list(emt_clean.columns) == final_emt_columns
assert emt_clean["id_emt"].notna().all()
assert emt_clean["id_emt"].is_unique
assert emt_clean["nombre"].notna().all()
assert emt_clean["latitud"].notna().all()
assert emt_clean["longitud"].notna().all()
assert emt_clean["es_parking_emt"].notna().all()
assert emt_clean["plazas_standard"].dropna().ge(0).all()
assert not emt_clean["plazas_standard"].eq(0).any()
assert emt_clean["plazas_pmr"].ge(0).all()

### 3.6. Resultado limpio

In [23]:
print("Shape:", emt_clean.shape)
display(emt_clean.head())

Shape: (87, 9)


,id_emt,nombre_original,nombre,direccion,latitud,longitud,es_parking_emt,plazas_standard,plazas_pmr
0,2,Colón,Colón,Plaza de Colón s/n,40.424709,-3.689939,False,1047,21
1,3,Corazón de María II,Corazón de María II,C/ Corazón de María (final calle),40.438525,-3.645525,False,327,0
2,4,Encuentro,Encuentro,Plaza Encuentro,40.405465,-3.651354,False,104,0
3,5,Nuestra Señora del Recuerdo,Nuestra Señora del Recuerdo,"c/ Hiedra, 26",40.472181,-3.679160,True,902,12
4,6,Corona Boreal,Corona Boreal,Plaza Corona Boreal y C/ Marqués Camarines,40.456800,-3.783200,False,120,0


### 3.7. Escritura controlada

In [24]:
if WRITE_OUTPUTS:
    EMT_INTERIM_PATH.parent.mkdir(parents=True, exist_ok=True)

    emt_clean.to_parquet(
        EMT_INTERIM_PATH,
        index=False,
    )

    written_emt = pd.read_parquet(EMT_INTERIM_PATH)

    assert list(written_emt.columns) == list(emt_clean.columns)
    assert len(written_emt) == len(emt_clean)
    assert written_emt["id_emt"].is_unique

    print("Output escrito:", EMT_INTERIM_PATH)
    print("Registros escritos:", len(written_emt))
else:
    print(
        "Escritura desactivada. "
        "Cambiar WRITE_OUTPUTS=True después de revisar el bloque EMT."
    )

Output escrito: /Users/hugo/TFM_parking_madrid/data/interim/emt/emt_parkings/emt_parkings_clean.parquet
Registros escritos: 87


## 4. Fuente `emt_aparcamientos_publicos`

### 4.1. Qué mide y para qué se utiliza

La fuente municipal contiene fichas estáticas de aparcamientos públicos, mixtos, disuasorios y de residentes. Incluye un identificador municipal, denominación, dirección, clasificación, coordenadas, distrito, barrio y campos descriptivos.

En este TFM se utiliza para:

- ampliar el inventario EMT con aparcamientos no presentes en `emt_parkings`;
- clasificar el tipo de acceso;
- incorporar distrito y barrio;
- contrastar nombres, direcciones y coordenadas;
- recuperar capacidades estructuradas desde `DESCRIPCION`.

### 4.2. Columnas conservadas y descartadas

Se conservan:

- `PK` → `pk_municipal`;
- `NOMBRE` → `nombre_original` y `nombre`;
- `TIPO` y el prefijo nominal → `tipo_acceso`;
- campos de vía y número → `direccion`;
- `LATITUD` y `LONGITUD`;
- códigos y nombres de distrito y barrio;
- `ACCESIBILIDAD`;
- `CONTENT-URL` → `url_ficha`;
- capacidades extraídas de `DESCRIPCION`.

Se descartan del output limpio:

- teléfono, fax y correo;
- transporte;
- equipamiento;
- planta, puerta y escalera;
- localidad y provincia;
- coordenadas UTM una vez comprobada la disponibilidad de WGS84;
- `DESCRIPCION` como texto libre, después de extraer los campos útiles;
- horario, debido a su escasa cobertura y formato no estructurado.

### 4.3. Comprobaciones previas

Las comprobaciones se limitan a:

- estructura y unicidad del identificador;
- nulos en campos esenciales;
- coordenadas inválidas o fuera de rango;
- dobles signos negativos en longitud;
- duplicados exactos y registros idénticos salvo `PK`;
- coordenadas exactamente repetidas;
- coherencia de códigos territoriales;
- valores de accesibilidad no reconocidos;
- clasificación del tipo de acceso;
- cobertura y posibles fallos de extracción desde `DESCRIPCION`.

In [25]:
municipal_raw = pd.read_csv(
    MUNICIPAL_RAW_PATH,
    encoding="latin1",
    sep=";",
)

expected_municipal_columns = [
    "PK",
    "NOMBRE",
    "DESCRIPCION-ENTIDAD",
    "HORARIO",
    "EQUIPAMIENTO",
    "TRANSPORTE",
    "DESCRIPCION",
    "ACCESIBILIDAD",
    "CONTENT-URL",
    "NOMBRE-VIA",
    "CLASE-VIAL",
    "TIPO-NUM",
    "NUM",
    "PLANTA",
    "PUERTA",
    "ESCALERAS",
    "ORIENTACION",
    "LOCALIDAD",
    "PROVINCIA",
    "CODIGO-POSTAL",
    "COD-BARRIO",
    "BARRIO",
    "COD-DISTRITO",
    "DISTRITO",
    "COORDENADA-X",
    "COORDENADA-Y",
    "LATITUD",
    "LONGITUD",
    "TELEFONO",
    "FAX",
    "EMAIL",
    "TIPO",
]

missing_municipal_columns = sorted(
    set(expected_municipal_columns)
    - set(municipal_raw.columns)
)

unexpected_municipal_columns = sorted(
    set(municipal_raw.columns)
    - set(expected_municipal_columns)
)

municipal_work = municipal_raw.copy()

longitude_text = (
    municipal_work["LONGITUD"]
    .astype("string")
    .str.strip()
)

double_negative_mask = longitude_text.str.match(
    r"^--\d",
    na=False,
)

municipal_work["longitud_corregida"] = (
    longitude_text
    .str.replace(r"^--", "-", regex=True)
)

municipal_work["longitud_num"] = pd.to_numeric(
    municipal_work["longitud_corregida"]
    .str.replace(",", ".", regex=False),
    errors="coerce",
)

municipal_work["latitud_num"] = pd.to_numeric(
    municipal_work["LATITUD"],
    errors="coerce",
)

municipal_work["cod_distrito_num"] = pd.to_numeric(
    municipal_work["COD-DISTRITO"],
    errors="raise",
).astype("Int64")

municipal_work["num_barrio"] = pd.to_numeric(
    municipal_work["COD-BARRIO"],
    errors="raise",
).astype("Int64")

# La fuente contiene un registro del barrio Justicia con código de
# distrito 13, aunque el distrito nominal es Centro. La referencia
# IVTM sitúa Justicia en distrito 1, barrio 4.
municipal_district_fix_mask = (
    municipal_work["DISTRITO"]
    .map(normalize_match_text)
    .eq("centro")
    &
    municipal_work["BARRIO"]
    .map(normalize_match_text)
    .eq("justicia")
    &
    municipal_work["cod_distrito_num"].eq(13)
    &
    municipal_work["num_barrio"].eq(4)
)

assert int(municipal_district_fix_mask.sum()) == 1, (
    "Se esperaba exactamente un registro municipal "
    "Centro/Justicia con código de distrito 13."
)

municipal_district_fix_pks = (
    municipal_work.loc[
        municipal_district_fix_mask,
        "PK",
    ]
    .astype(int)
    .tolist()
)

municipal_work.loc[
    municipal_district_fix_mask,
    "cod_distrito_num",
] = 1

municipal_work["cod_barrio"] = (
    municipal_work["cod_distrito_num"] * 100
    + municipal_work["num_barrio"]
).astype("Int64")

municipal_work["tipo_acceso"] = municipal_work.apply(
    lambda row: classify_municipal_access(
        row["NOMBRE"],
        row["TIPO"],
    ),
    axis=1,
).astype("string")

automobile_patterns = [
    r"n[úu]mero\s+de\s+plazas\s*:\s*([0-9][0-9\.,]*)",
    r"plazas?\s+totales?\s*:\s*([0-9][0-9\.,]*)",
    r"para\s+autom[oó]viles\s*:\s*([0-9][0-9\.,]*)",
    r"autom[oó]viles\s*:\s*([0-9][0-9\.,]*)",
    r"capacidad\s*:\s*([0-9][0-9\.,]*)\s+veh[ií]culos",
    r"plazas?\s*:\s*([0-9][0-9\.,]*)(?![0-9\.,])(?!\s*(?:residenciales?|residentes?|rotacionales?|p[úu]blicas?)\b)",
]

public_patterns = [
    r"([0-9][0-9\.\,\s]*)\s+"
    r"(?:plazas?\s+)?p[úu]blicas?",
    r"plazas?\s+p[úu]blicas?\s*:\s*"
    r"([0-9][0-9\.\,\s]*)",
    r"([0-9][0-9\.\,\s]*)\s+"
    r"(?:plazas?\s+)?de\s+rotaci[oó]n",
    r"rotaci[oó]n\s*:\s*"
    r"([0-9][0-9\.\,\s]*)",
    r"([0-9][0-9\.,]*)\s+rotacionales?",
]

resident_patterns = [
    r"([0-9][0-9\.\,\s]*)\s+"
    r"(?:plazas?\s+)?para\s+residentes",
    r"residentes\s*:\s*"
    r"([0-9][0-9\.\,\s]*)",
    r"([0-9][0-9\.,]*)\s+residenciales?",
]

pmr_patterns = [
    r"([0-9][0-9\.\,\s]*)\s+"
    r"para\s+(?:coches?|veh[ií]culos?)\s+"
    r"adaptados?(?:\s+PMR)?",
    r"([0-9][0-9\.\,\s]*)\s+"
    r"(?:plazas?\s+)?PMR",
    r"PMR\s*:\s*([0-9][0-9\.\,\s]*)",
]

electric_patterns = [
    r"([0-9][0-9\.\,\s]*)\s+"
    r"para\s+(?:coche|veh[ií]culo)s?\s+"
    r"el[eé]ctricos?",
    r"([0-9][0-9\.\,\s]*)\s+"
    r"(?:plazas?\s+)?(?:de\s+)?"
    r"recarga\s+el[eé]ctrica",
    r"el[eé]ctric[oa]s?\s*:\s*"
    r"([0-9][0-9\.\,\s]*)",
]

motorcycle_patterns = [
    r"para\s+motocicletas?\s*:\s*"
    r"([0-9][0-9\.\,\s]*)",
    r"([0-9][0-9\.\,\s]*)\s+"
    r"para\s+motocicletas?",
    r"plazas?\s+motos?\s*:\s*([0-9][0-9\.,]*)",
]

municipal_work["plazas_automoviles_desc"] = (
    municipal_work["DESCRIPCION"].map(
        lambda value: extract_first_count(
            value,
            automobile_patterns,
        )
    )
)

municipal_work["plazas_publicas_desc"] = (
    municipal_work["DESCRIPCION"].map(
        lambda value: extract_first_count(
            value,
            public_patterns,
        )
    )
)

municipal_work["plazas_residentes_desc"] = (
    municipal_work["DESCRIPCION"].map(
        lambda value: extract_first_count(
            value,
            resident_patterns,
        )
    )
)

municipal_work["plazas_pmr_desc"] = (
    municipal_work["DESCRIPCION"].map(
        lambda value: extract_first_count(
            value,
            pmr_patterns,
        )
    )
)

municipal_work["plazas_electricas_desc"] = (
    municipal_work["DESCRIPCION"].map(
        lambda value: extract_first_count(
            value,
            electric_patterns,
        )
    )
)

municipal_work["plazas_motocicletas_desc"] = (
    municipal_work["DESCRIPCION"].map(
        lambda value: extract_first_count(
            value,
            motorcycle_patterns,
        )
    )
)

capacity_columns = [
    "plazas_automoviles_desc",
    "plazas_publicas_desc",
    "plazas_residentes_desc",
    "plazas_pmr_desc",
    "plazas_electricas_desc",
    "plazas_motocicletas_desc",
]

for column in capacity_columns:
    municipal_work[column] = pd.to_numeric(
        municipal_work[column],
        errors="coerce",
    ).astype("Int64")

municipal_work["capacidad_extraida"] = (
    municipal_work[capacity_columns]
    .notna()
    .any(axis=1)
)

capacity_hint_mask = (
    municipal_work["DESCRIPCION"]
    .astype("string")
    .str.contains(
        r"plazas?|autom[oó]vil|residentes|"
        r"motocicleta|PMR|el[eé]ctric",
        case=False,
        regex=True,
        na=False,
    )
)

unparsed_capacity_records = (
    municipal_work.loc[
        capacity_hint_mask
        & ~municipal_work["capacidad_extraida"],
        ["PK", "NOMBRE"],
    ]
    .head(20)
    .apply(tuple, axis=1)
    .tolist()
)

content_columns_without_pk = [
    column
    for column in municipal_raw.columns
    if column != "PK"
]

content_duplicate_mask = municipal_raw.duplicated(
    subset=content_columns_without_pk,
    keep=False,
)

coordinate_duplicate_mask = municipal_work.duplicated(
    subset=["latitud_num", "longitud_num"],
    keep=False,
)

district_inconsistencies = (
    municipal_raw.groupby("COD-DISTRITO")["DISTRITO"]
    .nunique(dropna=False)
    .gt(1)
    .sum()
)

neighborhood_inconsistencies = (
    municipal_raw.groupby(
        ["COD-DISTRITO", "COD-BARRIO"]
    )["BARRIO"]
    .nunique(dropna=False)
    .gt(1)
    .sum()
)

invalid_accessibility_values = sorted(
    set(
        municipal_raw["ACCESIBILIDAD"]
        .dropna()
        .astype(int)
    )
    - set(range(0, 7))
)

source_type = (
    municipal_raw["TIPO"]
    .astype("string")
    .map(normalize_match_text)
)

name_type = (
    municipal_raw["NOMBRE"]
    .astype("string")
    .map(normalize_match_text)
)

source_name_conflict_mask = (
    source_type.str.contains("residentes", na=False)
    & (
        name_type.str.contains("publico", na=False)
        | name_type.str.contains("disuasorio", na=False)
    )
)

source_name_conflicts = (
    municipal_raw.loc[
        source_name_conflict_mask,
        ["PK", "NOMBRE", "TIPO"],
    ]
    .apply(tuple, axis=1)
    .tolist()
)

breakdown_complete_mask = (
    municipal_work["plazas_automoviles_desc"].notna()
    & municipal_work["plazas_publicas_desc"].notna()
    & municipal_work["plazas_residentes_desc"].notna()
)

breakdown_inconsistent_mask = (
    breakdown_complete_mask
    & (
        municipal_work["plazas_automoviles_desc"]
        != (
            municipal_work["plazas_publicas_desc"]
            + municipal_work["plazas_residentes_desc"]
        )
    )
)

inconsistent_breakdown_records = (
    municipal_work.loc[
        breakdown_inconsistent_mask,
        [
            "PK",
            "NOMBRE",
            "plazas_automoviles_desc",
            "plazas_publicas_desc",
            "plazas_residentes_desc",
        ],
    ]
    .apply(tuple, axis=1)
    .tolist()
)

municipal_checks = pd.DataFrame(
    [
        ("n_registros", len(municipal_raw)),
        ("n_columnas", len(municipal_raw.columns)),
        (
            "columnas_esperadas_ausentes",
            missing_municipal_columns,
        ),
        (
            "columnas_inesperadas",
            unexpected_municipal_columns,
        ),
        ("pk_nulos", int(municipal_raw["PK"].isna().sum())),
        (
            "pk_duplicados_extra",
            int(
                municipal_raw["PK"]
                .duplicated(keep="first")
                .sum()
            ),
        ),
        (
            "nombre_nulos",
            int(municipal_raw["NOMBRE"].isna().sum()),
        ),
        (
            "latitud_nulos",
            int(municipal_work["latitud_num"].isna().sum()),
        ),
        (
            "longitud_no_convertible_antes_correccion",
            int(
                pd.to_numeric(
                    longitude_text,
                    errors="coerce",
                ).isna().sum()
            ),
        ),
        (
            "longitudes_con_doble_signo",
            int(double_negative_mask.sum()),
        ),
        (
            "registros_longitud_doble_signo",
            municipal_raw.loc[
                double_negative_mask,
                ["PK", "NOMBRE", "LONGITUD"],
            ]
            .apply(tuple, axis=1)
            .tolist(),
        ),
        (
            "longitud_nulos_despues_correccion",
            int(
                municipal_work["longitud_num"]
                .isna()
                .sum()
            ),
        ),
        (
            "coordenadas_fuera_rango_amplio",
            int(
                (
                    ~municipal_work["latitud_num"]
                    .between(40.0, 41.0)
                    | ~municipal_work["longitud_num"]
                    .between(-4.5, -3.0)
                ).sum()
            ),
        ),
        (
            "filas_duplicadas_exactas_extra",
            int(
                municipal_raw
                .duplicated(keep="first")
                .sum()
            ),
        ),
        (
            "registros_duplicados_sin_pk_extra",
            int(
                municipal_raw
                .duplicated(
                    subset=content_columns_without_pk,
                    keep="first",
                )
                .sum()
            ),
        ),
        (
            "registros_duplicados_sin_pk",
            municipal_raw.loc[
                content_duplicate_mask,
                ["PK", "NOMBRE"],
            ]
            .apply(tuple, axis=1)
            .tolist(),
        ),
        (
            "grupos_coordenadas_exactas_repetidas",
            int(
                municipal_work.loc[
                    coordinate_duplicate_mask,
                    ["latitud_num", "longitud_num"],
                ]
                .drop_duplicates()
                .shape[0]
            ),
        ),
        (
            "registros_coordenadas_exactas_repetidas",
            municipal_work.loc[
                coordinate_duplicate_mask,
                ["PK", "NOMBRE"],
            ]
            .apply(tuple, axis=1)
            .tolist(),
        ),
        (
            "inconsistencias_codigo_nombre_distrito",
            int(district_inconsistencies),
        ),
        (
            "inconsistencias_clave_nombre_barrio",
            int(neighborhood_inconsistencies),
        ),
        (
            "accesibilidad_valores_no_reconocidos",
            invalid_accessibility_values,
        ),
        (
            "tipo_acceso_nulos",
            int(
                municipal_work["tipo_acceso"]
                .isna()
                .sum()
            ),
        ),
        (
            "tipo_acceso_conteos",
            municipal_work["tipo_acceso"]
            .value_counts(dropna=False)
            .to_dict(),
        ),
        (
            "conflictos_tipo_fuente_nombre",
            source_name_conflicts,
        ),
        (
            "descripcion_nulos",
            int(
                municipal_raw["DESCRIPCION"]
                .isna()
                .sum()
            ),
        ),
        (
            "plazas_automoviles_extraidas",
            int(
                municipal_work[
                    "plazas_automoviles_desc"
                ].notna().sum()
            ),
        ),
        (
            "plazas_publicas_extraidas",
            int(
                municipal_work[
                    "plazas_publicas_desc"
                ].notna().sum()
            ),
        ),
        (
            "plazas_residentes_extraidas",
            int(
                municipal_work[
                    "plazas_residentes_desc"
                ].notna().sum()
            ),
        ),
        (
            "plazas_pmr_extraidas",
            int(
                municipal_work[
                    "plazas_pmr_desc"
                ].notna().sum()
            ),
        ),
        (
            "plazas_electricas_extraidas",
            int(
                municipal_work[
                    "plazas_electricas_desc"
                ].notna().sum()
            ),
        ),
        (
            "plazas_motocicletas_extraidas",
            int(
                municipal_work[
                    "plazas_motocicletas_desc"
                ].notna().sum()
            ),
        ),
        (
            "registros_con_indicio_capacidad_sin_extraccion",
            int(
                (
                    capacity_hint_mask
                    & ~municipal_work["capacidad_extraida"]
                ).sum()
            ),
        ),
        (
            "muestra_indicio_capacidad_sin_extraccion",
            unparsed_capacity_records,
        ),
        (
    "desgloses_publicas_residentes_incompatibles",
    int(breakdown_inconsistent_mask.sum()),
),
(
    "registros_desglose_incompatible",
    inconsistent_breakdown_records,
),
    ],
    columns=["check", "valor"],
)

display(municipal_checks)

,check,valor
0,n_registros,63
1,n_columnas,32
2,columnas_esperadas_ausentes,[]
3,columnas_inesperadas,[]
4,pk_nulos,0
5,pk_duplicados_extra,0
6,nombre_nulos,0
7,latitud_nulos,0
8,longitud_no_convertible_antes_correccion,1
9,longitudes_con_doble_signo,1


### 4.4. Lectura de las comprobaciones y decisiones

El fichero contiene 63 registros y las 32 columnas esperadas. Los identificadores `PK` son completos y únicos, y no existen nombres o latitudes ausentes. Tampoco se detectan filas duplicadas, registros idénticos salvo por el identificador ni coordenadas exactamente repetidas, por lo que no se elimina ningún registro.

La única incidencia espacial corresponde al aparcamiento mixto Encuentro (`PK = 36269`), cuya longitud contiene un doble signo negativo. La corrección de `--3.651386...` a `-3.651386...` permite convertir correctamente la coordenada y la sitúa dentro del rango geográfico esperado, por lo que se aplica esta corrección determinista.

Los códigos y nombres de distrito y barrio mantienen correspondencias unívocas dentro de la fuente municipal. Como el fichero proporciona por separado el código de distrito y el número local del barrio, se construye provisionalmente `cod_barrio = cod_distrito × 100 + num_barrio`. Antes de adoptar esta variable como clave común, su correspondencia se contrasta en el apartado siguiente con la fuente IVTM.

La clasificación nominal identifica 39 aparcamientos mixtos, 19 públicos y 5 disuasorios, sin registros sin clasificar. La única discrepancia corresponde a Plaza de España, denominada aparcamiento público pero incluida en la categoría municipal de aparcamientos de residentes. Se clasifica provisionalmente como `publico` porque el nombre específico y la descripción reflejan uso público con plazas para abonados, no un uso exclusivamente residencial.

El campo `DESCRIPCION` está informado en los 63 registros. Tras adaptar las expresiones regulares a los distintos formatos textuales, se extrae la capacidad total cuando aparece expresamente indicada y, cuando existe un desglose, se conservan por separado las plazas públicas o rotacionales, residenciales, PMR, eléctricas y de motocicletas. No se infieren totales a partir de la suma de componentes cuando la fuente no proporciona una cifra total explícita.

### 4.5. Validación externa del código de barrio

La fuente municipal almacena por separado el código de distrito y el número local del barrio. Para comprobar si la variable derivada `cod_barrio = cod_distrito × 100 + num_barrio` coincide con la codificación utilizada en otras fuentes del TFM, se contrasta con el output limpio de IVTM.

Antes de comparar ambas fuentes se armonizan determinadas variantes nominales de barrio y se comprueba que cada `cod_barrio` esté asociado de forma unívoca a un distrito y a un nombre normalizado, tanto en IVTM como en la fuente municipal. Una vez descartadas ambigüedades internas, se realiza un cruce directo por `cod_barrio` y se comprueba que coincidan el distrito y el nombre normalizado del barrio.

El criterio de aceptación exige que todos los códigos municipales tengan correspondencia en IVTM, que no existan códigos ambiguos y que, para cada código compartido, coincidan el distrito y la entidad territorial representada. Los casos sin correspondencia o incompatibles se muestran explícitamente antes de aceptar la clave.

In [26]:
IVTM_BARRIOS_PATH = (
    ROOT
    / "data/interim/ser/ser_padron_vehiculos_ivtm_barrio"
    / "ser_padron_vehiculos_ivtm_barrio_clean.parquet"
)

assert IVTM_BARRIOS_PATH.exists(), (
    f"No existe la referencia IVTM: {IVTM_BARRIOS_PATH}"
)

ivtm_barrio_raw = pd.read_parquet(
    IVTM_BARRIOS_PATH
)

required_ivtm_columns = {
    "cod_distrito",
    "cod_barrio",
    "barrio",
}

missing_ivtm_columns = sorted(
    required_ivtm_columns
    - set(ivtm_barrio_raw.columns)
)

assert not missing_ivtm_columns, (
    "La referencia IVTM no contiene las columnas "
    f"territoriales esperadas: {missing_ivtm_columns}"
)


# Variantes nominales observadas que representan
# la misma entidad territorial.
BARRIO_NAME_ALIASES = {
    "pena grande": "penagrande",
    "villaverde alto casco historico de villaverde":
        "villaverde alto casco historico",
    "las aguilas": "aguilas",
}


def normalize_barrio_name(value):
    normalized = normalize_match_text(value)
    return BARRIO_NAME_ALIASES.get(
        normalized,
        normalized,
    )


# ---------------------------------------------------------------
# Referencia IVTM
# ---------------------------------------------------------------

ivtm_barrio_ref = (
    ivtm_barrio_raw[
        [
            "cod_distrito",
            "cod_barrio",
            "barrio",
        ]
    ]
    .dropna()
    .copy()
)

ivtm_barrio_ref["cod_distrito"] = pd.to_numeric(
    ivtm_barrio_ref["cod_distrito"],
    errors="raise",
).astype("Int64")

ivtm_barrio_ref["cod_barrio"] = pd.to_numeric(
    ivtm_barrio_ref["cod_barrio"],
    errors="raise",
).astype("Int64")

ivtm_barrio_ref["barrio_norm"] = (
    ivtm_barrio_ref["barrio"]
    .map(normalize_barrio_name)
)

ivtm_unique = (
    ivtm_barrio_ref[
        [
            "cod_barrio",
            "cod_distrito",
            "barrio_norm",
        ]
    ]
    .drop_duplicates()
)

ivtm_mapping_quality = (
    ivtm_unique
    .groupby("cod_barrio", as_index=False)
    .agg(
        n_distritos=(
            "cod_distrito",
            "nunique",
        ),
        n_nombres=(
            "barrio_norm",
            "nunique",
        ),
    )
)

ivtm_ambiguous_codes = (
    ivtm_mapping_quality.loc[
        ivtm_mapping_quality["n_distritos"].ne(1)
        | ivtm_mapping_quality["n_nombres"].ne(1),
        "cod_barrio",
    ]
    .tolist()
)

ivtm_map = (
    ivtm_unique
    .groupby("cod_barrio", as_index=False)
    .agg(
        cod_distrito_ivtm=(
            "cod_distrito",
            "first",
        ),
        barrio_ivtm_norm=(
            "barrio_norm",
            "first",
        ),
    )
)


# ---------------------------------------------------------------
# Referencia municipal corregida
# ---------------------------------------------------------------

municipal_barrio_ref = (
    municipal_work[
        [
            "cod_barrio",
            "cod_distrito_num",
            "BARRIO",
        ]
    ]
    .drop_duplicates()
    .copy()
)

municipal_barrio_ref[
    "barrio_municipal_norm"
] = (
    municipal_barrio_ref["BARRIO"]
    .map(normalize_barrio_name)
)

municipal_mapping_quality = (
    municipal_barrio_ref
    .groupby("cod_barrio", as_index=False)
    .agg(
        n_distritos=(
            "cod_distrito_num",
            "nunique",
        ),
        n_nombres=(
            "barrio_municipal_norm",
            "nunique",
        ),
    )
)

municipal_ambiguous_codes = (
    municipal_mapping_quality.loc[
        municipal_mapping_quality["n_distritos"].ne(1)
        | municipal_mapping_quality["n_nombres"].ne(1),
        "cod_barrio",
    ]
    .tolist()
)

municipal_map = (
    municipal_barrio_ref
    .groupby("cod_barrio", as_index=False)
    .agg(
        cod_distrito_municipal=(
            "cod_distrito_num",
            "first",
        ),
        barrio_municipal_norm=(
            "barrio_municipal_norm",
            "first",
        ),
    )
)


# ---------------------------------------------------------------
# Comparación directa por código canónico
# ---------------------------------------------------------------

territorial_comparison = municipal_map.merge(
    ivtm_map,
    on="cod_barrio",
    how="left",
    validate="one_to_one",
)

missing_ivtm_codes = (
    territorial_comparison.loc[
        territorial_comparison[
            "cod_distrito_ivtm"
        ].isna(),
        "cod_barrio",
    ]
    .astype(int)
    .tolist()
)

district_mismatch_codes = (
    territorial_comparison.loc[
        territorial_comparison[
            "cod_distrito_ivtm"
        ].notna()
        &
        (
            territorial_comparison[
                "cod_distrito_municipal"
            ]
            != territorial_comparison[
                "cod_distrito_ivtm"
            ]
        ),
        "cod_barrio",
    ]
    .astype(int)
    .tolist()
)

name_mismatch_codes = (
    territorial_comparison.loc[
        territorial_comparison[
            "barrio_ivtm_norm"
        ].notna()
        &
        (
            territorial_comparison[
                "barrio_municipal_norm"
            ]
            != territorial_comparison[
                "barrio_ivtm_norm"
            ]
        ),
        "cod_barrio",
    ]
    .astype(int)
    .tolist()
)

territorial_checks = pd.DataFrame(
    [
        (
            "registros_codigo_distrito_corregido",
            municipal_district_fix_pks,
        ),
        (
            "codigos_ivtm_ambiguos_tras_armonizacion",
            len(ivtm_ambiguous_codes),
        ),
        (
            "codigos_municipales_ambiguos",
            len(municipal_ambiguous_codes),
        ),
        (
            "codigos_municipales_sin_ivtm",
            len(missing_ivtm_codes),
        ),
        (
            "distritos_incompatibles",
            len(district_mismatch_codes),
        ),
        (
            "nombres_barrio_incompatibles",
            len(name_mismatch_codes),
        ),
    ],
    columns=["check", "valor"],
)

display(territorial_checks)

assert not ivtm_ambiguous_codes, (
    f"Códigos IVTM ambiguos: {ivtm_ambiguous_codes}"
)

assert not municipal_ambiguous_codes, (
    f"Códigos municipales ambiguos: "
    f"{municipal_ambiguous_codes}"
)

assert not missing_ivtm_codes, (
    f"Códigos municipales sin referencia IVTM: "
    f"{missing_ivtm_codes}"
)

assert not district_mismatch_codes, (
    f"Distritos incompatibles: "
    f"{district_mismatch_codes}"
)

assert not name_mismatch_codes, (
    f"Nombres de barrio incompatibles: "
    f"{name_mismatch_codes}"
)

,check,valor
0,registros_codigo_distrito_corregido,[52114]
1,codigos_ivtm_ambiguos_tras_armonizacion,0
2,codigos_municipales_ambiguos,0
3,codigos_municipales_sin_ivtm,0
4,distritos_incompatibles,0
5,nombres_barrio_incompatibles,0


#### Lectura de la validación territorial

La validación identificó un único registro cuya codificación territorial requirió corrección: el aparcamiento con `PK = 52114`, asociado al barrio Justicia. Aunque la fuente original le asigna el código de distrito 13, el nombre de distrito y la referencia IVTM lo sitúan en Centro, por lo que se corrigió a `cod_distrito = 1` y se derivó `cod_barrio = 104`.

Después de aplicar esta corrección y armonizar los nombres de barrio utilizados en la comparación, no se detectan códigos ambiguos, barrios municipales sin correspondencia en IVTM, discrepancias de distrito ni diferencias nominales entre ambas fuentes.

Por tanto, la expresión `cod_barrio = cod_distrito × 100 + num_barrio` queda validada para los registros de la fuente municipal y puede utilizarse como clave territorial común en las fases posteriores del TFM.

### 4.6. Limpieza

In [27]:
municipal_cod_distrito = (
    municipal_work["cod_distrito_num"]
    .astype("Int64")
)

municipal_num_barrio = (
    municipal_work["num_barrio"]
    .astype("Int64")
)

municipal_cod_barrio = (
    municipal_work["cod_barrio"]
    .astype("Int64")
)

municipal_clean = pd.DataFrame(
    {
        "pk_municipal": pd.to_numeric(
            municipal_work["PK"],
            errors="raise",
        ).astype("Int64"),

        "nombre_original": (
            municipal_work["NOMBRE"]
            .astype("string")
        ),

        "nombre": (
            municipal_work["NOMBRE"]
            .map(clean_parking_name)
            .astype("string")
        ),

        "tipo_acceso": (
            municipal_work["tipo_acceso"]
            .astype("string")
        ),

        "direccion": (
            municipal_work
            .apply(
                build_municipal_address,
                axis=1,
            )
            .astype("string")
        ),

        "latitud": (
            municipal_work["latitud_num"]
            .astype("float64")
        ),

        "longitud": (
            municipal_work["longitud_num"]
            .astype("float64")
        ),

        "cod_distrito": municipal_cod_distrito,

        "distrito": (
            municipal_work["DISTRITO"]
            .map(clean_visible_text)
            .astype("string")
        ),

        "num_barrio": municipal_num_barrio,

        "cod_barrio": municipal_cod_barrio,

        "barrio": (
            municipal_work["BARRIO"]
            .map(clean_visible_text)
            .astype("string")
        ),

        "accesibilidad": pd.to_numeric(
            municipal_work["ACCESIBILIDAD"],
            errors="raise",
        ).astype("Int64"),

        "url_ficha": (
            municipal_work["CONTENT-URL"]
            .map(clean_visible_text)
            .astype("string")
        ),

        "plazas_automoviles_desc": (
            municipal_work[
                "plazas_automoviles_desc"
            ].astype("Int64")
        ),

        "plazas_publicas_desc": (
            municipal_work[
                "plazas_publicas_desc"
            ].astype("Int64")
        ),

        "plazas_residentes_desc": (
            municipal_work[
                "plazas_residentes_desc"
            ].astype("Int64")
        ),

        "plazas_pmr_desc": (
            municipal_work[
                "plazas_pmr_desc"
            ].astype("Int64")
        ),

        "plazas_electricas_desc": (
            municipal_work[
                "plazas_electricas_desc"
            ].astype("Int64")
        ),

        "plazas_motocicletas_desc": (
            municipal_work[
                "plazas_motocicletas_desc"
            ].astype("Int64")
        ),
    }
)

final_municipal_columns = [
    "pk_municipal",
    "nombre_original",
    "nombre",
    "tipo_acceso",
    "direccion",
    "latitud",
    "longitud",
    "cod_distrito",
    "distrito",
    "num_barrio",
    "cod_barrio",
    "barrio",
    "accesibilidad",
    "url_ficha",
    "plazas_automoviles_desc",
    "plazas_publicas_desc",
    "plazas_residentes_desc",
    "plazas_pmr_desc",
    "plazas_electricas_desc",
    "plazas_motocicletas_desc",
]

municipal_clean = municipal_clean[
    final_municipal_columns
]

# Comprobaciones silenciosas del contrato.
assert len(municipal_clean) == len(municipal_raw)
assert list(municipal_clean.columns) == final_municipal_columns
assert municipal_clean["pk_municipal"].notna().all()
assert municipal_clean["pk_municipal"].is_unique
assert municipal_clean["nombre"].notna().all()
assert municipal_clean["tipo_acceso"].notna().all()
assert municipal_clean["latitud"].notna().all()
assert municipal_clean["longitud"].notna().all()
assert municipal_clean["latitud"].between(40.0, 41.0).all()
assert municipal_clean["longitud"].between(-4.5, -3.0).all()
assert municipal_clean["cod_distrito"].notna().all()
assert municipal_clean["cod_barrio"].notna().all()

### 4.7. Resultado limpio

In [28]:
print("Shape:", municipal_clean.shape)
display(municipal_clean.head())

Shape: (63, 20)


,pk_municipal,nombre_original,nombre,tipo_acceso,direccion,latitud,longitud,cod_distrito,distrito,num_barrio,cod_barrio,barrio,accesibilidad,url_ficha,plazas_automoviles_desc,plazas_publicas_desc,plazas_residentes_desc,plazas_pmr_desc,plazas_electricas_desc,plazas_motocicletas_desc
0,11483771,Aparcamiento disuasorio Aviación Española,Aviación Española,disuasorio,"Calle Fuente de Lima, 5",40.383238,-3.783621,10,LATINA,7,1007,LAS AGUILAS,1,http://www.madrid.es/sites/v/index.jsp?vgnextchannel=bfa48ab43d6bb410VgnVCM100000171f5a0aRCRD&vgnextoid=1ec084bc6763...,344,<NA>,<NA>,<NA>,10,25
1,10489514,Aparcamiento disuasorio Estadio Metropolitano Sur ES-02b,Estadio Metropolitano Sur ES-02b,disuasorio,"Avenida Arcentales, 37",40.434154,-3.598831,20,SAN BLAS-CANILLEJAS,5,2005,ROSAS,0,http://www.madrid.es/sites/v/index.jsp?vgnextchannel=bfa48ab43d6bb410VgnVCM100000171f5a0aRCRD&vgnextoid=975b89a80c49...,3011,<NA>,<NA>,<NA>,<NA>,61
2,11413444,Aparcamiento disuasorio Fuente de la Mora,Fuente de la Mora,disuasorio,"Avenida Manoteras, 1",40.484443,-3.664503,16,HORTALEZA,6,1606,VALDEFUENTES,1,http://www.madrid.es/sites/v/index.jsp?vgnextchannel=bfa48ab43d6bb410VgnVCM100000171f5a0aRCRD&vgnextoid=cbb3ed735e9f...,368,<NA>,<NA>,12,10,35
3,11544464,Aparcamiento disuasorio Islazul,Islazul,disuasorio,"Calle Calderilla, 1",40.365217,-3.737923,11,CARABANCHEL,6,1106,BUENAVISTA,1,http://www.madrid.es/sites/v/index.jsp?vgnextchannel=bfa48ab43d6bb410VgnVCM100000171f5a0aRCRD&vgnextoid=83bf252aa267...,224,<NA>,<NA>,<NA>,<NA>,<NA>
4,11413620,Aparcamiento disuasorio Pitis,Pitis,disuasorio,Calle Gloria Fuertes,40.494193,-3.727236,8,FUENCARRAL-EL PARDO,2,802,FUENTELARREINA,1,http://www.madrid.es/sites/v/index.jsp?vgnextchannel=bfa48ab43d6bb410VgnVCM100000171f5a0aRCRD&vgnextoid=1642e9887f9f...,402,<NA>,<NA>,12,10,34


### 4.8. Escritura controlada

In [29]:
if WRITE_OUTPUTS:
    MUNICIPAL_INTERIM_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    municipal_clean.to_parquet(
        MUNICIPAL_INTERIM_PATH,
        index=False,
    )

    written_municipal = pd.read_parquet(
        MUNICIPAL_INTERIM_PATH
    )

    assert list(written_municipal.columns) == list(
        municipal_clean.columns
    )
    assert len(written_municipal) == len(municipal_clean)
    assert written_municipal["pk_municipal"].is_unique

    print("Output escrito:", MUNICIPAL_INTERIM_PATH)
    print("Registros escritos:", len(written_municipal))
else:
    print(
        "Escritura desactivada. "
        "Cambiar WRITE_OUTPUTS=True después de revisar "
        "el bloque municipal."
    )

Output escrito: /Users/hugo/TFM_parking_madrid/data/interim/emt/emt_aparcamientos_publicos/emt_aparcamientos_publicos_clean.parquet
Registros escritos: 63


## 5. Integración de fuentes

### 5.1. Contrato de matching

El objetivo de este bloque es determinar qué registros de `emt_parkings` y `emt_aparcamientos_publicos` representan un mismo aparcamiento físico, evitando duplicidades y uniones forzadas.

La unidad inicial de análisis es el par candidato `id_emt`–`pk_municipal`. Cada par representa una hipótesis de correspondencia y no una fila del inventario final.

El output analítico principal será `inventario_global_emt`, con una fila por aparcamiento físico e identificador único `parking_uid`. Los identificadores EMT y municipales originales se conservarán para permitir la trazabilidad y los joins posteriores con las fuentes de ocupación.

Cuando una misma entidad física quede asociada a varios identificadores de origen, se utilizará una relación técnica auxiliar para conservarlos. Esta relación no constituye una segunda salida analítica ni se mostrará como resultado principal del notebook.

La resolución combina:

- distancia geográfica;
- coincidencia exacta de nombres normalizados;
- similitud secuencial del nombre;
- solapamiento y contención de tokens;
- similitud de direcciones;
- reciprocidad de los mejores candidatos;
- capacidad como comprobación secundaria;
- revisión explícita de casos conflictivos.

Ningún registro se fusiona únicamente por proximidad. Las correspondencias no respaldadas por evidencia suficiente se conservarán como registros exclusivos de su fuente.

### 5.2. Generación de candidatos

Las fuentes contienen 87 registros EMT y 63 registros municipales. Se construye inicialmente una matriz completa de `87 × 63 = 5.481` pares posibles. Cada fila representa una hipótesis de correspondencia entre un identificador EMT y un identificador municipal, no un aparcamiento del inventario final.

La preselección conserva la unión de los mejores pares por distancia, similitud secuencial del nombre y similitud de la dirección, considerando ambas direcciones del cruce. La contención nominal y el índice de Jaccard se mantienen como señales de evaluación, pero no se utilizan mediante rankings para seleccionar candidatos porque generan numerosos empates entre pares sin solapamiento nominal.

Si todas las relaciones fueran estrictamente uno-a-uno y no existieran duplicidades internas, el número de entidades resultaría de `87 + 63 - n_matches`. Esta expresión se conserva únicamente como referencia teórica. La validación posterior debe permitir que varios identificadores de origen representen una misma entidad física, siempre que exista evidencia suficiente y la decisión quede documentada.

In [30]:
from difflib import SequenceMatcher


GENERIC_PARKING_TOKENS = {
    "aparcamiento",
    "parking",
    "publico",
    "publica",
    "mixto",
    "disuasorio",
    "emt",
    "municipal",
}


def normalize_parking_match_name(value):
    normalized = normalize_match_text(value)

    tokens = [
        token
        for token in normalized.split()
        if token not in GENERIC_PARKING_TOKENS
    ]

    return " ".join(tokens)


def sequence_similarity(left, right):
    if not left or not right:
        return np.nan

    return SequenceMatcher(
        None,
        left,
        right,
    ).ratio()


def token_jaccard(left, right):
    left_tokens = set(str(left).split())
    right_tokens = set(str(right).split())

    if not left_tokens or not right_tokens:
        return np.nan

    return (
        len(left_tokens & right_tokens)
        / len(left_tokens | right_tokens)
    )


def token_containment(left, right):
    left_tokens = set(str(left).split())
    right_tokens = set(str(right).split())

    if not left_tokens or not right_tokens:
        return np.nan

    return (
        len(left_tokens & right_tokens)
        / min(
            len(left_tokens),
            len(right_tokens),
        )
    )


def haversine_distance_m(
    lat_1,
    lon_1,
    lat_2,
    lon_2,
):
    earth_radius_m = 6_371_008.8

    lat_1_rad = np.radians(lat_1)
    lon_1_rad = np.radians(lon_1)
    lat_2_rad = np.radians(lat_2)
    lon_2_rad = np.radians(lon_2)

    delta_lat = lat_2_rad - lat_1_rad
    delta_lon = lon_2_rad - lon_1_rad

    haversine_value = (
        np.sin(delta_lat / 2) ** 2
        + np.cos(lat_1_rad)
        * np.cos(lat_2_rad)
        * np.sin(delta_lon / 2) ** 2
    )

    return (
        2
        * earth_radius_m
        * np.arcsin(
            np.sqrt(haversine_value)
        )
    )


emt_matching = (
    emt_clean[
        [
            "id_emt",
            "nombre",
            "direccion",
            "latitud",
            "longitud",
            "plazas_standard",
            "plazas_pmr",
        ]
    ]
    .rename(
        columns={
            "nombre": "nombre_emt",
            "direccion": "direccion_emt",
            "latitud": "latitud_emt",
            "longitud": "longitud_emt",
            "plazas_standard": "plazas_standard_emt",
            "plazas_pmr": "plazas_pmr_emt",
        }
    )
    .copy()
)

municipal_matching = (
    municipal_clean[
        [
            "pk_municipal",
            "nombre",
            "direccion",
            "latitud",
            "longitud",
            "tipo_acceso",
            "cod_barrio",
            "barrio",
            "plazas_automoviles_desc",
            "plazas_publicas_desc",
            "plazas_residentes_desc",
        ]
    ]
    .rename(
        columns={
            "nombre": "nombre_municipal",
            "direccion": "direccion_municipal",
            "latitud": "latitud_municipal",
            "longitud": "longitud_municipal",
            "barrio": "barrio_municipal",
        }
    )
    .copy()
)

emt_matching["nombre_match_emt"] = (
    emt_matching["nombre_emt"]
    .map(normalize_parking_match_name)
)

emt_matching["direccion_match_emt"] = (
    emt_matching["direccion_emt"]
    .map(normalize_match_text)
)

municipal_matching["nombre_match_municipal"] = (
    municipal_matching["nombre_municipal"]
    .map(normalize_parking_match_name)
)

municipal_matching["direccion_match_municipal"] = (
    municipal_matching["direccion_municipal"]
    .map(normalize_match_text)
)

emt_matching["_cross_key"] = 1
municipal_matching["_cross_key"] = 1

pair_matrix = (
    emt_matching
    .merge(
        municipal_matching,
        on="_cross_key",
        how="inner",
        validate="many_to_many",
    )
    .drop(columns="_cross_key")
)

pair_matrix["distancia_m"] = haversine_distance_m(
    pair_matrix["latitud_emt"],
    pair_matrix["longitud_emt"],
    pair_matrix["latitud_municipal"],
    pair_matrix["longitud_municipal"],
)

pair_matrix["nombre_exacto"] = (
    pair_matrix["nombre_match_emt"].ne("")
    &
    (
        pair_matrix["nombre_match_emt"]
        == pair_matrix["nombre_match_municipal"]
    )
)

pair_matrix["similitud_nombre"] = [
    sequence_similarity(left, right)
    for left, right in zip(
        pair_matrix["nombre_match_emt"],
        pair_matrix["nombre_match_municipal"],
    )
]

pair_matrix["jaccard_nombre"] = [
    token_jaccard(left, right)
    for left, right in zip(
        pair_matrix["nombre_match_emt"],
        pair_matrix["nombre_match_municipal"],
    )
]

pair_matrix["contencion_nombre"] = [
    token_containment(left, right)
    for left, right in zip(
        pair_matrix["nombre_match_emt"],
        pair_matrix["nombre_match_municipal"],
    )
]

pair_matrix["similitud_direccion"] = [
    sequence_similarity(left, right)
    for left, right in zip(
        pair_matrix["direccion_match_emt"],
        pair_matrix["direccion_match_municipal"],
    )
]

pair_matrix["n_tokens_nombre_emt"] = (
    pair_matrix["nombre_match_emt"]
    .str.split()
    .map(len)
)

pair_matrix["n_tokens_nombre_municipal"] = (
    pair_matrix["nombre_match_municipal"]
    .str.split()
    .map(len)
)


# Rankings desde la perspectiva EMT.
pair_matrix["rank_distancia_emt"] = (
    pair_matrix
    .groupby("id_emt")["distancia_m"]
    .rank(
        method="min",
        ascending=True,
    )
)

pair_matrix["rank_nombre_emt"] = (
    pair_matrix
    .groupby("id_emt")["similitud_nombre"]
    .rank(
        method="min",
        ascending=False,
    )
)


pair_matrix["rank_direccion_emt"] = (
    pair_matrix
    .groupby("id_emt")["similitud_direccion"]
    .rank(
        method="min",
        ascending=False,
    )
)


# Rankings desde la perspectiva municipal.
pair_matrix["rank_distancia_municipal"] = (
    pair_matrix
    .groupby("pk_municipal")["distancia_m"]
    .rank(
        method="min",
        ascending=True,
    )
)

pair_matrix["rank_nombre_municipal"] = (
    pair_matrix
    .groupby("pk_municipal")["similitud_nombre"]
    .rank(
        method="min",
        ascending=False,
    )
)


pair_matrix["rank_direccion_municipal"] = (
    pair_matrix
    .groupby("pk_municipal")["similitud_direccion"]
    .rank(
        method="min",
        ascending=False,
    )
)


candidate_mask = (
    pair_matrix["nombre_exacto"]
    | pair_matrix["rank_distancia_emt"].le(3)
    | pair_matrix["rank_nombre_emt"].le(3)
    | pair_matrix["rank_direccion_emt"].le(2)
    | pair_matrix["rank_distancia_municipal"].le(3)
    | pair_matrix["rank_nombre_municipal"].le(3)
    | pair_matrix["rank_direccion_municipal"].le(2)
)

candidate_pairs = (
    pair_matrix.loc[candidate_mask]
    .sort_values(
        [
            "id_emt",
            "rank_distancia_emt",
            "rank_nombre_emt",
            "pk_municipal",
        ]
    )
    .reset_index(drop=True)
)

assert len(pair_matrix) == (
    len(emt_clean)
    * len(municipal_clean)
)

assert set(candidate_pairs["id_emt"]) == set(
    emt_clean["id_emt"]
)

assert set(candidate_pairs["pk_municipal"]) == set(
    municipal_clean["pk_municipal"]
)

### 5.3. Diagnóstico de los candidatos

La generación amplia debe mantener cobertura completa de ambas fuentes sin convertir automáticamente cada mejor candidato en una correspondencia real.

El diagnóstico utiliza:

- coincidencia exacta del nombre normalizado;
- similitud secuencial;
- solapamiento de tokens mediante Jaccard;
- contención del nombre más corto en el más largo;
- distancia geográfica;
- similitud de dirección;
- reciprocidad de los mejores candidatos.

La contención permite reconocer denominaciones abreviadas o ampliadas, como `América` frente a `Avenida de América`, sin asumir todavía que constituyen un match definitivo.

Los identificadores EMT 104 y 105 se excluyen únicamente de la calibración de distancias porque presentan coordenadas anómalas ya identificadas. Se conservan en la matriz de candidatos y deberán resolverse mediante el resto de las señales.

In [31]:
exact_name_pairs = pair_matrix.loc[
    pair_matrix["nombre_exacto"]
].copy()

full_containment_pairs = pair_matrix.loc[
    pair_matrix["contencion_nombre"].eq(1)
].copy()

mutual_distance_pairs = pair_matrix.loc[
    pair_matrix["rank_distancia_emt"].eq(1)
    & pair_matrix["rank_distancia_municipal"].eq(1)
].copy()

mutual_sequence_pairs = pair_matrix.loc[
    pair_matrix["rank_nombre_emt"].eq(1)
    & pair_matrix["rank_nombre_municipal"].eq(1)
].copy()


# ---------------------------------------------------------------
# Candidato principal por distancia
# ---------------------------------------------------------------

top_distance_emt = (
    pair_matrix
    .sort_values(
        [
            "id_emt",
            "distancia_m",
            "pk_municipal",
        ]
    )
    .groupby(
        "id_emt",
        sort=False,
    )
    .head(1)
    [
        [
            "id_emt",
            "nombre_emt",
            "pk_municipal",
            "nombre_municipal",
            "direccion_municipal",
            "distancia_m",
            "similitud_nombre",
            "contencion_nombre",
            "similitud_direccion",
        ]
    ]
    .rename(
        columns={
            "pk_municipal":
                "pk_top_distancia",
            "nombre_municipal":
                "municipal_top_distancia",
            "direccion_municipal":
                "direccion_top_distancia",
            "distancia_m":
                "distancia_top_m",
            "similitud_nombre":
                "sim_nombre_top_distancia",
            "contencion_nombre":
                "contencion_top_distancia",
            "similitud_direccion":
                "sim_direccion_top_distancia",
        }
    )
)


# ---------------------------------------------------------------
# Candidato principal por evidencia nominal
# ---------------------------------------------------------------

top_nominal_emt = (
    pair_matrix
    .sort_values(
        [
            "id_emt",
            "nombre_exacto",
            "contencion_nombre",
            "similitud_nombre",
            "jaccard_nombre",
            "distancia_m",
            "pk_municipal",
        ],
        ascending=[
            True,
            False,
            False,
            False,
            False,
            True,
            True,
        ],
    )
    .groupby(
        "id_emt",
        sort=False,
    )
    .head(1)
    [
        [
            "id_emt",
            "pk_municipal",
            "nombre_municipal",
            "direccion_municipal",
            "distancia_m",
            "nombre_exacto",
            "similitud_nombre",
            "jaccard_nombre",
            "contencion_nombre",
            "similitud_direccion",
        ]
    ]
    .rename(
        columns={
            "pk_municipal":
                "pk_top_nominal",
            "nombre_municipal":
                "municipal_top_nominal",
            "direccion_municipal":
                "direccion_top_nominal",
            "distancia_m":
                "distancia_nominal_m",
            "nombre_exacto":
                "top_nominal_exacto",
            "similitud_nombre":
                "sim_nombre_top_nominal",
            "jaccard_nombre":
                "jaccard_top_nominal",
            "contencion_nombre":
                "contencion_top_nominal",
            "similitud_direccion":
                "sim_direccion_top_nominal",
        }
    )
)

top1_comparison = top_distance_emt.merge(
    top_nominal_emt,
    on="id_emt",
    how="inner",
    validate="one_to_one",
)

top1_comparison["mismo_candidato"] = (
    top1_comparison["pk_top_distancia"]
    == top1_comparison["pk_top_nominal"]
)

top1_disagreements = (
    top1_comparison.loc[
        ~top1_comparison["mismo_candidato"]
    ]
    .copy()
)


# ---------------------------------------------------------------
# Calibración espacial robusta
# ---------------------------------------------------------------

KNOWN_COORDINATE_ANOMALIES = {
    104,
    105,
}

exact_name_reference_pairs = (
    exact_name_pairs.loc[
        ~exact_name_pairs["id_emt"].isin(
            KNOWN_COORDINATE_ANOMALIES
        )
    ]
    .copy()
)

if exact_name_reference_pairs.empty:
    reference_distance_p90 = np.nan
else:
    reference_distance_p90 = (
        exact_name_reference_pairs[
            "distancia_m"
        ]
        .quantile(0.90)
    )

exact_name_distance_summary = pd.DataFrame(
    [
        (
            "n_pares_nombre_exacto",
            len(exact_name_pairs),
        ),
        (
            "n_pares_calibracion_sin_anomalias",
            len(exact_name_reference_pairs),
        ),
        (
            "distancia_min_m",
            (
                round(
                    exact_name_reference_pairs[
                        "distancia_m"
                    ].min(),
                    1,
                )
                if not exact_name_reference_pairs.empty
                else pd.NA
            ),
        ),
        (
            "distancia_mediana_m",
            (
                round(
                    exact_name_reference_pairs[
                        "distancia_m"
                    ].median(),
                    1,
                )
                if not exact_name_reference_pairs.empty
                else pd.NA
            ),
        ),
        (
            "distancia_p90_m",
            (
                round(
                    reference_distance_p90,
                    1,
                )
                if not pd.isna(reference_distance_p90)
                else pd.NA
            ),
        ),
        (
            "distancia_max_m",
            (
                round(
                    exact_name_reference_pairs[
                        "distancia_m"
                    ].max(),
                    1,
                )
                if not exact_name_reference_pairs.empty
                else pd.NA
            ),
        ),
    ],
    columns=[
        "metrica",
        "valor",
    ],
)


# ---------------------------------------------------------------
# Casos relevantes para revisión
# ---------------------------------------------------------------

top1_comparison["evidencia_nominal_fuerte"] = (
    top1_comparison["top_nominal_exacto"]
    | top1_comparison[
        "contencion_top_nominal"
    ].ge(0.95)
    | top1_comparison[
        "sim_nombre_top_nominal"
    ].ge(0.85)
)

if pd.isna(reference_distance_p90):
    top1_comparison[
        "proximidad_en_rango_referencia"
    ] = False
else:
    top1_comparison[
        "proximidad_en_rango_referencia"
    ] = (
        top1_comparison[
            "distancia_top_m"
        ].le(reference_distance_p90)
        | top1_comparison[
            "distancia_nominal_m"
        ].le(reference_distance_p90)
    )

review_mask = (
    ~top1_comparison["mismo_candidato"]
    &
    (
        top1_comparison["evidencia_nominal_fuerte"]
        | top1_comparison[
            "proximidad_en_rango_referencia"
        ]
        | top1_comparison["id_emt"].isin(
            KNOWN_COORDINATE_ANOMALIES
        )
    )
)

review_cases_full = (
    top1_comparison.loc[
        review_mask
    ]
    .sort_values(
        [
            "top_nominal_exacto",
            "contencion_top_nominal",
            "sim_nombre_top_nominal",
            "distancia_nominal_m",
        ],
        ascending=[
            False,
            False,
            False,
            True,
        ],
    )
    .copy()
)

review_cases_display = (
    review_cases_full[
        [
            "id_emt",
            "nombre_emt",
            "pk_top_distancia",
            "municipal_top_distancia",
            "distancia_top_m",
            "pk_top_nominal",
            "municipal_top_nominal",
            "distancia_nominal_m",
            "top_nominal_exacto",
            "contencion_top_nominal",
            "sim_nombre_top_nominal",
            "sim_direccion_top_nominal",
        ]
    ]
    .head(20)
    .copy()
)

for column in [
    "distancia_top_m",
    "distancia_nominal_m",
]:
    review_cases_display[column] = (
        review_cases_display[column]
        .round(1)
    )

for column in [
    "contencion_top_nominal",
    "sim_nombre_top_nominal",
    "sim_direccion_top_nominal",
]:
    review_cases_display[column] = (
        review_cases_display[column]
        .round(3)
    )


# ---------------------------------------------------------------
# Checks del diagnóstico
# ---------------------------------------------------------------

candidate_generation_checks = pd.DataFrame(
    [
        (
            "pares_evaluados",
            len(pair_matrix),
        ),
        (
            "pares_candidatos_conservados",
            len(candidate_pairs),
        ),
        (
            "registros_emt_cubiertos",
            candidate_pairs[
                "id_emt"
            ].nunique(),
        ),
        (
            "registros_municipales_cubiertos",
            candidate_pairs[
                "pk_municipal"
            ].nunique(),
        ),
        (
            "pares_nombre_exacto",
            len(exact_name_pairs),
        ),
        (
            "pares_contencion_nombre_completa",
            len(full_containment_pairs),
        ),
        (
            "pares_mejor_distancia_reciproca",
            len(mutual_distance_pairs),
        ),
        (
            "pares_mejor_secuencia_reciproca",
            len(mutual_sequence_pairs),
        ),
        (
            "emt_con_mismo_top_distancia_y_nominal",
            int(
                top1_comparison[
                    "mismo_candidato"
                ].sum()
            ),
        ),
        (
            "emt_con_top_distancia_y_nominal_distintos",
            len(top1_disagreements),
        ),
        (
            "discrepancias_relevantes_para_revision",
            len(review_cases_full),
        ),
        (
            "discrepancias_mostradas",
            len(review_cases_display),
        ),
    ],
    columns=[
        "check",
        "valor",
    ],
)

display(candidate_generation_checks)

print(
    "Distancias de calibración para pares con "
    "nombre exacto, excluyendo IDs 104 y 105:"
)

display(
    exact_name_distance_summary
)

print(
    "Casos relevantes para revisión "
    "(máximo 20; tabla diagnóstica provisional):"
)

if review_cases_display.empty:
    print(
        "No se identifican casos relevantes."
    )
else:
    display(
        review_cases_display
    )

,check,valor
0,pares_evaluados,5481
1,pares_candidatos_conservados,728
2,registros_emt_cubiertos,87
3,registros_municipales_cubiertos,63
4,pares_nombre_exacto,51
5,pares_contencion_nombre_completa,63
6,pares_mejor_distancia_reciproca,54
7,pares_mejor_secuencia_reciproca,57
8,emt_con_mismo_top_distancia_y_nominal,56
9,emt_con_top_distancia_y_nominal_distintos,31


Distancias de calibración para pares con nombre exacto, excluyendo IDs 104 y 105:


,metrica,valor
0,n_pares_nombre_exacto,51.0
1,n_pares_calibracion_sin_anomalias,50.0
2,distancia_min_m,2.1
3,distancia_mediana_m,61.1
4,distancia_p90_m,353.8
5,distancia_max_m,1008.4


Casos relevantes para revisión (máximo 20; tabla diagnóstica provisional):


,id_emt,nombre_emt,pk_top_distancia,municipal_top_distancia,distancia_top_m,pk_top_nominal,municipal_top_nominal,distancia_nominal_m,top_nominal_exacto,contencion_top_nominal,sim_nombre_top_nominal,sim_direccion_top_nominal
43,51,Museo de la ciudad,7393089,Auditorio Nacional de Música (Príncipe de Vergara),356.5,88141,Museo de la Ciudad,364.0,True,1.000,1.000,0.697
33,40,Serrano III,13465,Montalbán,278.0,7417052,Serrano III,529.7,True,1.000,1.000,0.328
36,43,Velázquez Jorge Juan,13464,Felipe II,327.3,13457,Velázquez - Jorge Juan,617.7,True,1.000,1.000,0.558
84,105,Arquitecto Ribera,10489514,Estadio Metropolitano Sur ES-02b,11809.9,13470,Arquitecto Ribera,20350.5,True,1.000,1.000,NaN
82,101,SANTO DOMINGO,5898885,Avenida de Portugal,2934.1,11544439,Plaza de Santo Domingo,5300.5,False,1.000,0.743,0.927
23,28,General Yagüe,5517342,Orense,141.1,13515,San Germán II (antes General Yagüe),236.7,False,1.000,0.565,0.150
77,95,Colón,7417052,Serrano III,6.6,13454,Plaza de Colón,146.7,False,1.000,0.526,0.200
0,2,Colón,7417052,Serrano III,39.7,13454,Plaza de Colón,148.4,False,1.000,0.526,0.200
38,45,Auditorio,13457,Velázquez - Jorge Juan,389.0,7393089,Auditorio Nacional de Música (Príncipe de Vergara),2383.1,False,1.000,0.316,0.250
83,104,Velázquez-JuanBravo,10489514,Estadio Metropolitano Sur ES-02b,11809.9,13468,Velázquez - Juan Bravo,18822.9,False,0.500,0.974,NaN


#### 5.3.1. Validación de las coincidencias nominales exactas

La coincidencia exacta se calcula después de normalizar los nombres y eliminar términos genéricos relacionados con el estacionamiento. Esta señal constituye una evidencia nominal fuerte, pero no se adopta automáticamente como correspondencia definitiva.

Antes de utilizarla se comprueba:

- que cada identificador EMT participe como máximo en un par exacto;
- que cada identificador municipal participe como máximo en un par exacto;
- qué pares exactos superan el percentil 90 de la distancia de referencia.

Los pares espacialmente alejados se revisan mediante dirección, capacidad y demás señales disponibles. Los identificadores EMT 104 y 105 permanecen excluidos de esta calibración por sus coordenadas anómalas conocidas.

In [32]:
# ---------------------------------------------------------------
# Unicidad de las coincidencias nominales exactas
# ---------------------------------------------------------------

exact_matches_per_emt = (
    exact_name_pairs
    .groupby("id_emt")
    .size()
)

exact_matches_per_municipal = (
    exact_name_pairs
    .groupby("pk_municipal")
    .size()
)

emt_with_multiple_exact_matches = (
    exact_matches_per_emt.loc[
        exact_matches_per_emt.gt(1)
    ]
    .index
    .astype(int)
    .tolist()
)

municipal_with_multiple_exact_matches = (
    exact_matches_per_municipal.loc[
        exact_matches_per_municipal.gt(1)
    ]
    .index
    .astype(int)
    .tolist()
)


# ---------------------------------------------------------------
# Pares exactos alejados respecto a la referencia espacial
# ---------------------------------------------------------------

if pd.isna(reference_distance_p90):
    exact_pairs_above_p90 = (
        exact_name_reference_pairs
        .iloc[0:0]
        .copy()
    )
else:
    exact_pairs_above_p90 = (
        exact_name_reference_pairs.loc[
            exact_name_reference_pairs[
                "distancia_m"
            ].gt(reference_distance_p90)
        ]
        .sort_values(
            "distancia_m",
            ascending=False,
        )
        .copy()
    )

exact_match_validation_checks = pd.DataFrame(
    [
        (
            "pares_nombre_exacto",
            len(exact_name_pairs),
        ),
        (
            "id_emt_con_multiples_pares_exactos",
            len(emt_with_multiple_exact_matches),
        ),
        (
            "pk_municipal_con_multiples_pares_exactos",
            len(municipal_with_multiple_exact_matches),
        ),
        (
            "pares_exactos_usados_en_calibracion",
            len(exact_name_reference_pairs),
        ),
        (
            "pares_exactos_sobre_p90_distancia",
            len(exact_pairs_above_p90),
        ),
    ],
    columns=[
        "check",
        "valor",
    ],
)

display(
    exact_match_validation_checks
)


if emt_with_multiple_exact_matches:
    print(
        "IDs EMT con más de una coincidencia exacta:",
        emt_with_multiple_exact_matches,
    )

if municipal_with_multiple_exact_matches:
    print(
        "PK municipales con más de una coincidencia exacta:",
        municipal_with_multiple_exact_matches,
    )


exact_pairs_above_p90_display = (
    exact_pairs_above_p90[
        [
            "id_emt",
            "nombre_emt",
            "direccion_emt",
            "pk_municipal",
            "nombre_municipal",
            "direccion_municipal",
            "distancia_m",
            "similitud_direccion",
            "plazas_standard_emt",
            "plazas_automoviles_desc",
            "plazas_publicas_desc",
            "plazas_residentes_desc",
        ]
    ]
    .copy()
)

if not exact_pairs_above_p90_display.empty:
    exact_pairs_above_p90_display[
        "distancia_m"
    ] = (
        exact_pairs_above_p90_display[
            "distancia_m"
        ]
        .round(1)
    )

    exact_pairs_above_p90_display[
        "similitud_direccion"
    ] = (
        exact_pairs_above_p90_display[
            "similitud_direccion"
        ]
        .round(3)
    )

    print(
        "Pares nominalmente exactos situados "
        "por encima del percentil 90 de distancia:"
    )

    display(
        exact_pairs_above_p90_display
    )
else:
    print(
        "No existen pares nominalmente exactos "
        "por encima del percentil 90 de distancia."
    )

,check,valor
0,pares_nombre_exacto,51
1,id_emt_con_multiples_pares_exactos,0
2,pk_municipal_con_multiples_pares_exactos,1
3,pares_exactos_usados_en_calibracion,50
4,pares_exactos_sobre_p90_distancia,5


PK municipales con más de una coincidencia exacta: [88141]
Pares nominalmente exactos situados por encima del percentil 90 de distancia:


,id_emt,nombre_emt,direccion_emt,pk_municipal,nombre_municipal,direccion_municipal,distancia_m,similitud_direccion,plazas_standard_emt,plazas_automoviles_desc,plazas_publicas_desc,plazas_residentes_desc
323,7,Avenida de Portugal,"Avda. de Portugal, s/n. Frente al nº 51",5898885,Avenida de Portugal,"Avenida Portugal, 155",1008.4,0.536,435,<NA>,428,445
2310,43,Velázquez Jorge Juan,C/ Velázquez ( y C/ Jorge Juan),13457,Velázquez - Jorge Juan,"Calle Velazquez, 1",617.7,0.558,549,<NA>,548,395
2118,40,Serrano III,C/ Serrano entre Jorge Juan y Pza. Independencia,7417052,Serrano III,"Calle Serrano, 2",529.7,0.328,261,<NA>,261,725
3847,76,Pitis,"Calle Gloria Fuertes, 191",11413620,Pitis,Calle Gloria Fuertes,380.5,0.909,461,402,<NA>,<NA>
2761,51,Museo de la ciudad,"C/ Príncipe de Vergara, 152 ( y C/ Pechuán )",88141,Museo de la Ciudad,"Calle Principe de Vergara, 134",364.0,0.697,147,147,<NA>,<NA>


#### 5.3.2. Nombres EMT repetidos y relaciones varios-a-uno

La validación de correspondencias exactas muestra que un mismo registro municipal puede recibir varios candidatos EMT nominalmente idénticos. Esto puede deberse a duplicidades históricas, identificadores operativos distintos, accesos diferentes o instalaciones realmente separadas que comparten denominación.

Antes de construir `parking_uid`, se revisan los nombres EMT normalizados repetidos, la distancia entre sus registros, sus capacidades y sus mejores candidatos municipales.

La coincidencia nominal no basta para consolidar dos identificadores EMT. La decisión debe estar respaldada por coordenadas, dirección, capacidad, clasificación EMT y relación con la fuente municipal.

In [33]:
from itertools import combinations


# ---------------------------------------------------------------
# Nombres EMT normalizados repetidos
# ---------------------------------------------------------------

emt_name_reference = (
    emt_matching[
        [
            "id_emt",
            "nombre_match_emt",
        ]
    ]
    .drop_duplicates()
)

emt_name_counts = (
    emt_name_reference
    .groupby("nombre_match_emt")
    .size()
)

repeated_name_mask = emt_name_counts.gt(1)

repeated_name_mask = (
    repeated_name_mask
    & pd.Series(
        emt_name_counts.index != "",
        index=emt_name_counts.index,
    )
)

repeated_emt_names = (
    emt_name_counts.loc[
        repeated_name_mask
    ]
    .index
    .tolist()
)

repeated_emt_records = (
    emt_clean
    .merge(
        emt_name_reference,
        on="id_emt",
        how="left",
        validate="one_to_one",
    )
    .loc[
        lambda dataframe:
            dataframe["nombre_match_emt"].isin(
                repeated_emt_names
            )
    ]
    .copy()
)


# ---------------------------------------------------------------
# Distancia interna dentro de cada nombre repetido
# ---------------------------------------------------------------

intra_emt_distance_rows = []

for normalized_name, group in repeated_emt_records.groupby(
    "nombre_match_emt",
    sort=True,
):
    records = group.to_dict("records")

    for left, right in combinations(records, 2):
        distance = haversine_distance_m(
            left["latitud"],
            left["longitud"],
            right["latitud"],
            right["longitud"],
        )

        intra_emt_distance_rows.append(
            {
                "nombre_match_emt": normalized_name,
                "id_emt_1": left["id_emt"],
                "id_emt_2": right["id_emt"],
                "distancia_entre_ids_m": round(
                    float(distance),
                    1,
                ),
                "plazas_standard_1":
                    left["plazas_standard"],
                "plazas_standard_2":
                    right["plazas_standard"],
                "es_parking_emt_1":
                    left["es_parking_emt"],
                "es_parking_emt_2":
                    right["es_parking_emt"],
            }
        )

intra_emt_distances = pd.DataFrame(
    intra_emt_distance_rows
)


# ---------------------------------------------------------------
# Candidatos principales de los registros repetidos
# ---------------------------------------------------------------

repeated_emt_diagnostic = (
    repeated_emt_records
    .merge(
        top1_comparison[
            [
                "id_emt",
                "pk_top_distancia",
                "municipal_top_distancia",
                "distancia_top_m",
                "pk_top_nominal",
                "municipal_top_nominal",
                "distancia_nominal_m",
                "top_nominal_exacto",
                "contencion_top_nominal",
                "sim_nombre_top_nominal",
            ]
        ],
        on="id_emt",
        how="left",
        validate="one_to_one",
    )
    [
        [
            "nombre_match_emt",
            "id_emt",
            "nombre",
            "direccion",
            "latitud",
            "longitud",
            "es_parking_emt",
            "plazas_standard",
            "pk_top_distancia",
            "municipal_top_distancia",
            "distancia_top_m",
            "pk_top_nominal",
            "municipal_top_nominal",
            "distancia_nominal_m",
            "top_nominal_exacto",
            "contencion_top_nominal",
            "sim_nombre_top_nominal",
        ]
    ]
    .sort_values(
        [
            "nombre_match_emt",
            "id_emt",
        ]
    )
    .copy()
)

for column in [
    "distancia_top_m",
    "distancia_nominal_m",
]:
    repeated_emt_diagnostic[column] = (
        repeated_emt_diagnostic[column]
        .round(1)
    )

for column in [
    "contencion_top_nominal",
    "sim_nombre_top_nominal",
]:
    repeated_emt_diagnostic[column] = (
        repeated_emt_diagnostic[column]
        .round(3)
    )


# ---------------------------------------------------------------
# Conflictos exactos sobre un mismo PK municipal
# ---------------------------------------------------------------

duplicated_exact_pks = (
    exact_name_pairs
    .groupby("pk_municipal")
    .size()
    .loc[lambda series: series.gt(1)]
    .index
    .tolist()
)

exact_conflict_pairs = (
    exact_name_pairs.loc[
        exact_name_pairs[
            "pk_municipal"
        ].isin(duplicated_exact_pks)
    ]
    .merge(
        emt_clean[
            [
                "id_emt",
                "es_parking_emt",
            ]
        ],
        on="id_emt",
        how="left",
        validate="many_to_one",
    )
    [
        [
            "id_emt",
            "nombre_emt",
            "direccion_emt",
            "es_parking_emt",
            "plazas_standard_emt",
            "pk_municipal",
            "nombre_municipal",
            "direccion_municipal",
            "plazas_automoviles_desc",
            "plazas_publicas_desc",
            "distancia_m",
            "similitud_direccion",
        ]
    ]
    .sort_values(
        [
            "pk_municipal",
            "distancia_m",
        ]
    )
    .copy()
)

if not exact_conflict_pairs.empty:
    exact_conflict_pairs["distancia_m"] = (
        exact_conflict_pairs["distancia_m"]
        .round(1)
    )

    exact_conflict_pairs[
        "similitud_direccion"
    ] = (
        exact_conflict_pairs[
            "similitud_direccion"
        ]
        .round(3)
    )


# ---------------------------------------------------------------
# Checks
# ---------------------------------------------------------------

repeated_name_checks = pd.DataFrame(
    [
        (
            "grupos_nombres_emt_repetidos",
            len(repeated_emt_names),
        ),
        (
            "registros_emt_implicados",
            len(repeated_emt_records),
        ),
        (
            "pk_municipales_con_conflicto_exacto",
            len(duplicated_exact_pks),
        ),
        (
            "pares_exactos_en_conflicto",
            len(exact_conflict_pairs),
        ),
    ],
    columns=[
        "check",
        "valor",
    ],
)

display(
    repeated_name_checks
)

print(
    "Registros EMT con nombres normalizados repetidos:"
)

display(
    repeated_emt_diagnostic
)

print(
    "Distancia entre identificadores EMT "
    "con la misma denominación:"
)

display(
    intra_emt_distances
)

print(
    "Pares exactos que compiten por "
    "el mismo identificador municipal:"
)

if exact_conflict_pairs.empty:
    print(
        "No se identifican conflictos exactos."
    )
else:
    display(
        exact_conflict_pairs
    )

,check,valor
0,grupos_nombres_emt_repetidos,2
1,registros_emt_implicados,4
2,pk_municipales_con_conflicto_exacto,1
3,pares_exactos_en_conflicto,2


Registros EMT con nombres normalizados repetidos:


,nombre_match_emt,id_emt,nombre,direccion,latitud,longitud,es_parking_emt,plazas_standard,pk_top_distancia,municipal_top_distancia,distancia_top_m,pk_top_nominal,municipal_top_nominal,distancia_nominal_m,top_nominal_exacto,contencion_top_nominal,sim_nombre_top_nominal
0,colon,2,Colón,Plaza de Colón s/n,40.424709,-3.689939,False,1047,7417052,Serrano III,39.7,13454,Plaza de Colón,148.4,False,1.0,0.526
2,colon,95,Colón,"Plaza de Colón, s/n",40.424419,-3.689574,True,4,7417052,Serrano III,6.6,13454,Plaza de Colón,146.7,False,1.0,0.526
1,museo de la ciudad,51,Museo de la ciudad,"C/ Príncipe de Vergara, 152 ( y C/ Pechuán )",40.447497,-3.677927,False,147,7393089,Auditorio Nacional de Música (Príncipe de Vergara),356.5,88141,Museo de la Ciudad,364.0,True,1.0,1.000
3,museo de la ciudad,99,Museo de la ciudad,c/ Cartagena 178,40.444212,-3.677665,True,146,88141,Museo de la Ciudad,2.1,88141,Museo de la Ciudad,2.1,True,1.0,1.000


Distancia entre identificadores EMT con la misma denominación:


,nombre_match_emt,id_emt_1,id_emt_2,distancia_entre_ids_m,plazas_standard_1,plazas_standard_2,es_parking_emt_1,es_parking_emt_2
0,colon,2,95,44.7,1047,4,False,True
1,museo de la ciudad,51,99,365.9,147,146,False,True


Pares exactos que compiten por el mismo identificador municipal:


,id_emt,nombre_emt,direccion_emt,es_parking_emt,plazas_standard_emt,pk_municipal,nombre_municipal,direccion_municipal,plazas_automoviles_desc,plazas_publicas_desc,distancia_m,similitud_direccion
1,99,Museo de la ciudad,c/ Cartagena 178,True,146,88141,Museo de la Ciudad,"Calle Principe de Vergara, 134",147,<NA>,2.1,0.318
0,51,Museo de la ciudad,"C/ Príncipe de Vergara, 152 ( y C/ Pechuán )",False,147,88141,Museo de la Ciudad,"Calle Principe de Vergara, 134",147,<NA>,364.0,0.697


### 5.4. Reglas de resolución y revisión final

Las coincidencias nominales exactas, únicas y sin incidencias constituyen el conjunto de aceptación automática inicial.

Se excluyen de esa aceptación automática:

- los pares exactos que compiten por un mismo identificador municipal;
- los pares situados por encima del percentil 90 de distancia;
- los identificadores EMT 104 y 105, cuyas coordenadas son anómalas;
- cualquier relación varios-a-uno que requiera consolidar más de un identificador de origen.

Los candidatos no exactos solo se consideran para revisión cuando combinan varias señales independientes. En concreto, deben presentar proximidad dentro de la distribución de referencia y, además, evidencia nominal fuerte, reciprocidad o una dirección compatible.

Los umbrales utilizados en este apartado no determinan automáticamente un match. Su única función es reducir la revisión final a un conjunto pequeño y trazable. Las decisiones excepcionales se documentarán en el código que construya el inventario, sin crear una tabla persistente independiente de decisiones.

In [34]:
# ---------------------------------------------------------------
# Coincidencias exactas inicialmente automatizables
# ---------------------------------------------------------------

manual_exact_ids = set(
    exact_pairs_above_p90["id_emt"]
    .astype(int)
    .tolist()
)

manual_exact_ids.update(
    KNOWN_COORDINATE_ANOMALIES
)

conflict_exact_pks = set(
    duplicated_exact_pks
)

automatic_exact_matches = (
    exact_name_pairs.loc[
        ~exact_name_pairs["id_emt"].isin(
            manual_exact_ids
        )
        &
        ~exact_name_pairs["pk_municipal"].isin(
            conflict_exact_pks
        )
    ]
    .copy()
)


# ---------------------------------------------------------------
# Señales para candidatos no exactos
# ---------------------------------------------------------------

final_review_pairs = candidate_pairs.copy()

final_review_pairs["reciproco_distancia"] = (
    final_review_pairs["rank_distancia_emt"].eq(1)
    & final_review_pairs[
        "rank_distancia_municipal"
    ].eq(1)
)

final_review_pairs["reciproco_nombre"] = (
    final_review_pairs["rank_nombre_emt"].eq(1)
    & final_review_pairs[
        "rank_nombre_municipal"
    ].eq(1)
)

final_review_pairs["reciproco_direccion"] = (
    final_review_pairs["rank_direccion_emt"].eq(1)
    & final_review_pairs[
        "rank_direccion_municipal"
    ].eq(1)
)

final_review_pairs["evidencia_nominal_fuerte"] = (
    final_review_pairs[
        "contencion_nombre"
    ].eq(1)
    |
    final_review_pairs[
        "similitud_nombre"
    ].ge(0.85)
)

final_review_pairs["direccion_compatible"] = (
    final_review_pairs[
        "similitud_direccion"
    ].ge(0.70)
)

if pd.isna(reference_distance_p90):
    final_review_pairs[
        "distancia_en_referencia"
    ] = False
else:
    final_review_pairs[
        "distancia_en_referencia"
    ] = (
        final_review_pairs[
            "distancia_m"
        ].le(reference_distance_p90)
    )

final_review_pairs[
    "pk_con_nombre_exacto"
] = (
    final_review_pairs[
        "pk_municipal"
    ].isin(
        exact_name_pairs[
            "pk_municipal"
        ]
    )
)

final_review_pairs[
    "id_emt_con_nombre_exacto"
] = (
    final_review_pairs[
        "id_emt"
    ].isin(
        exact_name_pairs[
            "id_emt"
        ]
    )
)


# ---------------------------------------------------------------
# Candidatos no exactos con combinación suficiente de señales
# ---------------------------------------------------------------

strong_non_exact_mask = (
    ~final_review_pairs["nombre_exacto"]
    &
    ~final_review_pairs["id_emt"].isin(
        KNOWN_COORDINATE_ANOMALIES
    )
    &
    final_review_pairs["distancia_en_referencia"]
    &
    (
        (
            final_review_pairs[
                "contencion_nombre"
            ].eq(1)
            &
            (
                final_review_pairs[
                    "reciproco_distancia"
                ]
                |
                final_review_pairs[
                    "reciproco_nombre"
                ]
                |
                final_review_pairs[
                    "direccion_compatible"
                ]
            )
        )
        |
        (
            final_review_pairs[
                "similitud_nombre"
            ].ge(0.85)
            &
            final_review_pairs[
                "reciproco_distancia"
            ]
        )
        |
        (
            final_review_pairs[
                "reciproco_distancia"
            ]
            &
            final_review_pairs[
                "reciproco_nombre"
            ]
        )
    )
)

strong_non_exact_candidates = (
    final_review_pairs.loc[
        strong_non_exact_mask
    ]
    .merge(
        emt_clean[
            [
                "id_emt",
                "es_parking_emt",
            ]
        ],
        on="id_emt",
        how="left",
        validate="many_to_one",
    )
    .sort_values(
        [
            "pk_con_nombre_exacto",
            "contencion_nombre",
            "similitud_nombre",
            "distancia_m",
        ],
        ascending=[
            True,
            False,
            False,
            True,
        ],
    )
    .copy()
)


# ---------------------------------------------------------------
# Conflictos dentro de la revisión final
# ---------------------------------------------------------------

strong_ids_with_multiple_candidates = (
    strong_non_exact_candidates
    .groupby("id_emt")
    .size()
    .loc[lambda series: series.gt(1)]
    .index
    .astype(int)
    .tolist()
)

strong_pks_with_multiple_candidates = (
    strong_non_exact_candidates
    .groupby("pk_municipal")
    .size()
    .loc[lambda series: series.gt(1)]
    .index
    .astype(int)
    .tolist()
)


# ---------------------------------------------------------------
# Tabla compacta visible
# ---------------------------------------------------------------

strong_non_exact_display = (
    strong_non_exact_candidates[
        [
            "id_emt",
            "nombre_emt",
            "direccion_emt",
            "es_parking_emt",
            "plazas_standard_emt",
            "pk_municipal",
            "nombre_municipal",
            "direccion_municipal",
            "tipo_acceso",
            "plazas_automoviles_desc",
            "plazas_publicas_desc",
            "plazas_residentes_desc",
            "distancia_m",
            "similitud_nombre",
            "contencion_nombre",
            "similitud_direccion",
            "reciproco_distancia",
            "reciproco_nombre",
            "reciproco_direccion",
            "pk_con_nombre_exacto",
        ]
    ]
    .copy()
)

for column in [
    "distancia_m",
]:
    strong_non_exact_display[column] = (
        strong_non_exact_display[column]
        .round(1)
    )

for column in [
    "similitud_nombre",
    "contencion_nombre",
    "similitud_direccion",
]:
    strong_non_exact_display[column] = (
        strong_non_exact_display[column]
        .round(3)
    )


final_review_checks = pd.DataFrame(
    [
        (
            "pares_exactos_totales",
            len(exact_name_pairs),
        ),
        (
            "pares_exactos_automatizables",
            len(automatic_exact_matches),
        ),
        (
            "pares_exactos_requieren_revision",
            (
                len(exact_name_pairs)
                - len(automatic_exact_matches)
            ),
        ),
        (
            "candidatos_no_exactos_fuertes",
            len(strong_non_exact_candidates),
        ),
        (
            "id_emt_con_varios_candidatos_fuertes",
            len(strong_ids_with_multiple_candidates),
        ),
        (
            "pk_con_varios_candidatos_fuertes",
            len(strong_pks_with_multiple_candidates),
        ),
        (
            "candidatos_fuertes_sobre_pk_con_match_exacto",
            int(
                strong_non_exact_candidates[
                    "pk_con_nombre_exacto"
                ].sum()
            ),
        ),
    ],
    columns=[
        "check",
        "valor",
    ],
)

display(
    final_review_checks
)

if strong_ids_with_multiple_candidates:
    print(
        "IDs EMT con varios candidatos no exactos fuertes:",
        strong_ids_with_multiple_candidates,
    )

if strong_pks_with_multiple_candidates:
    print(
        "PK municipales con varios candidatos no exactos fuertes:",
        strong_pks_with_multiple_candidates,
    )

print(
    "Candidatos no exactos que requieren "
    "una decisión final:"
)

if strong_non_exact_display.empty:
    print(
        "No se identifican candidatos no exactos fuertes."
    )
else:
    display(
        strong_non_exact_display
    )

,check,valor
0,pares_exactos_totales,51
1,pares_exactos_automatizables,44
2,pares_exactos_requieren_revision,7
3,candidatos_no_exactos_fuertes,9
4,id_emt_con_varios_candidatos_fuertes,0
5,pk_con_varios_candidatos_fuertes,0
6,candidatos_fuertes_sobre_pk_con_match_exacto,1


Candidatos no exactos que requieren una decisión final:


,id_emt,nombre_emt,direccion_emt,es_parking_emt,plazas_standard_emt,pk_municipal,nombre_municipal,direccion_municipal,tipo_acceso,plazas_automoviles_desc,plazas_publicas_desc,plazas_residentes_desc,distancia_m,similitud_nombre,contencion_nombre,similitud_direccion,reciproco_distancia,reciproco_nombre,reciproco_direccion,pk_con_nombre_exacto
4,46,Cortes,Plaza de las Cortes,False,942,13467,Las Cortes,"Plaza Cortes, 13",publico,480,<NA>,<NA>,48.3,0.750,1.0,0.706,True,True,False,False
5,64,Puerta Toledo,Glorieta Puerta Toledo C/V Cptan. Salazar Martínez,False,244,13478,Glorieta Puerta de Toledo,Glorieta Puerta de Toledo,publico,240,<NA>,<NA>,96.0,0.684,1.0,0.595,True,True,False,False
7,77,PLAZA DE SANTA ANA,PZA. DE SANTA ANA,False,328,13456,Santa Ana,Plaza Santa Ana,publico,325,<NA>,6,7.2,0.667,1.0,0.839,True,False,True,False
0,9,Paseo de Recoletos,"Paseo de Recoletos, 4",True,281,13476,Recoletos,"Paseo Recoletos, 4",mixto,<NA>,102,289,83.0,0.667,1.0,0.919,True,True,True,False
3,28,General Yagüe,"Avda. Brasil, 15 Nº15 al 19",False,409,13515,San Germán II (antes General Yagüe),"Calle Orense, 48",mixto,<NA>,419,<NA>,236.7,0.565,1.0,0.150,False,True,False,False
6,71,Pedro Zerolo,"Plaza Pedro Zerolo, s/n",True,115,52114,Pedro Zerolo 'antes denominado Vázquez de Mella',"Plaza Pedro Zerolo, 1",mixto,<NA>,107,261,17.1,0.414,1.0,0.905,True,False,True,False
1,16,América,Avda. América (intercambiador),False,269,59060,Avenida de América (intercambiador),"Avenida America, 9",mixto,<NA>,261,395,15.4,0.350,1.0,0.591,True,False,False,False
8,80,Wanda Metropolitano,"Avda. Arcentales, 39",True,3027,10489514,Estadio Metropolitano Sur ES-02b,"Avenida Arcentales, 37",disuasorio,3011,<NA>,<NA>,169.3,0.627,0.5,0.872,True,True,True,False
2,17,Arquitecto Ribera (Barceló),"C/ Barceló, 2",False,317,13470,Arquitecto Ribera,"Calle Barcelo, 2",mixto,<NA>,318,298,17.1,0.810,1.0,0.846,True,False,True,True


### 5.5. Resolución de correspondencias

La resolución definitiva combina 44 coincidencias nominales exactas automatizables con 17 decisiones manuales respaldadas por la revisión de nombres, direcciones, distancias y capacidades.

Se incorporan dos correspondencias adicionales que habían quedado fuera de la primera resolución por presentar coordenadas EMT incompatibles con el resto de las señales:

- `id_emt = 101` con `pk_municipal = 11544439` para Santo Domingo;
- `id_emt = 45` con `pk_municipal = 7393089` para el Auditorio Nacional.

La correspondencia del Auditorio Nacional no fue generada por la preselección inicial de candidatos, debido a la inconsistencia de sus coordenadas y a que las similitudes nominal y de dirección no superaban los criterios generales. Se incorpora como una excepción manual posterior a la auditoría residual, respaldada por la referencia común al Auditorio Nacional y por la coincidencia exacta de la capacidad de 376 plazas. Esta excepción se codifica explícitamente sin modificar los umbrales generales de generación de candidatos.

En ambos casos se adoptan las coordenadas municipales, igual que en el resto de entidades presentes en dicha fuente.

Las decisiones manuales prevalecen frente a las automáticas cuando comparten un identificador EMT o municipal. La prioridad se aplica de forma genérica y se valida antes de construir el inventario.

Los IDs EMT 95, 104, 105 y 69 se conservan como identificadores adicionales:

- `95` se asocia al ID `2` en la entidad Colón;
- `104` se asocia al ID `58` en Velázquez–Juan Bravo;
- `105` se asocia al ID `17` en Arquitecto Ribera;
- `69` se asocia también al ID `17`, al interpretarse Barceló y Arquitecto Ribera como componentes operativos de una misma instalación física.

Las capacidades de los alias no se suman. Los IDs 104 y 105 no se emplean como registros principales porque presentan dirección ausente, capacidad no informada y coordenadas anómalas.

Se rechazan explícitamente las relaciones Altamiras–Alcántara, Canalejas 360–Sevilla, Museo de la Ciudad `51`–`88141` y Barco 1–Arquitecto Ribera por falta de evidencia suficiente o por existir una correspondencia principal más consistente.

In [35]:
# ===============================================================
# RESOLUCIÓN DEFINITIVA DEL MATCHING
# ===============================================================

automatic_resolution = (
    automatic_exact_matches[
        [
            "id_emt",
            "pk_municipal",
        ]
    ]
    .copy()
)

automatic_resolution[
    "metodo_match"
] = "nombre_exacto_auto"

automatic_resolution[
    "motivo_match"
] = (
    "Nombre normalizado exacto, relación única "
    "y ausencia de incidencias."
)


# ---------------------------------------------------------------
# Decisiones manuales revisadas
# ---------------------------------------------------------------

manual_match_specs = [{'id_emt': 7,
  'pk_municipal': 5898885,
  'metodo_match': 'revision_exacta',
  'motivo_match': 'Avenida de Portugal: nombre exacto y capacidades públicas compatibles.'},
 {'id_emt': 43,
  'pk_municipal': 13457,
  'metodo_match': 'revision_exacta',
  'motivo_match': 'Velázquez-Jorge Juan: nombre exacto y 549 frente a 548 plazas públicas.'},
 {'id_emt': 40,
  'pk_municipal': 7417052,
  'metodo_match': 'revision_exacta',
  'motivo_match': 'Serrano III: nombre exacto y 261 plazas en ambas fuentes.'},
 {'id_emt': 76,
  'pk_municipal': 11413620,
  'metodo_match': 'revision_exacta',
  'motivo_match': 'Pitis: nombre exacto y misma vía.'},
 {'id_emt': 99,
  'pk_municipal': 88141,
  'metodo_match': 'conflicto_exacto_resuelto',
  'motivo_match': 'Museo de la Ciudad: distancia de 2,1 m y capacidades de 146 y 147 plazas.'},
 {'id_emt': 46,
  'pk_municipal': 13467,
  'metodo_match': 'revision_multisenal',
  'motivo_match': 'Cortes-Las Cortes: nombre contenido, misma plaza y reciprocidad.'},
 {'id_emt': 64,
  'pk_municipal': 13478,
  'metodo_match': 'revision_multisenal',
  'motivo_match': 'Puerta de Toledo: nombre, dirección y capacidades compatibles.'},
 {'id_emt': 77,
  'pk_municipal': 13456,
  'metodo_match': 'revision_multisenal',
  'motivo_match': 'Santa Ana: misma plaza, 7,2 m y capacidades compatibles.'},
 {'id_emt': 9,
  'pk_municipal': 13476,
  'metodo_match': 'revision_multisenal',
  'motivo_match': 'Recoletos: misma dirección y mejor candidato recíproco.'},
 {'id_emt': 28,
  'pk_municipal': 13515,
  'metodo_match': 'revision_multisenal',
  'motivo_match': 'Cambio nominal documentado de General Yagüe a San Germán II.'},
 {'id_emt': 71,
  'pk_municipal': 52114,
  'metodo_match': 'revision_multisenal',
  'motivo_match': 'Pedro Zerolo: misma plaza, 17,1 m y dirección compatible.'},
 {'id_emt': 16,
  'pk_municipal': 59060,
  'metodo_match': 'revision_multisenal',
  'motivo_match': 'América-Avenida de América: mismo intercambiador, 15,4 m y capacidad '
                  'compatible.'},
 {'id_emt': 80,
  'pk_municipal': 10489514,
  'metodo_match': 'revision_multisenal',
  'motivo_match': 'Wanda-Estadio Metropolitano: misma avenida y capacidades compatibles.'},
 {'id_emt': 17,
  'pk_municipal': 13470,
  'metodo_match': 'revision_multisenal',
  'motivo_match': 'Arquitecto Ribera-Barceló: misma dirección, 17,1 m y 317 frente a 318 '
                  'plazas públicas.'},
 {'id_emt': 2,
  'pk_municipal': 13454,
  'metodo_match': 'consolidacion_colon',
  'motivo_match': 'Colón: misma denominación y dirección; el ID EMT 95 se conserva como '
                  'alias.'},
 {'id_emt': 101,
  'pk_municipal': 11544439,
  'metodo_match': 'anomalia_espacial_revisada',
  'motivo_match': 'Santo Domingo: misma plaza, dirección altamente compatible y capacidades de '
                  '333 y 320 plazas. Se adoptan coordenadas municipales.'},
 {'id_emt': 45,
  'pk_municipal': 7393089,
  'metodo_match': 'anomalia_espacial_revisada',
  'motivo_match': 'Auditorio Nacional: correspondencia semántica, referencia a Príncipe de '
                  'Vergara y capacidad exacta de 376 plazas. Se adoptan coordenadas '
                  'municipales.'}]

manual_resolution = pd.DataFrame(
    manual_match_specs
)

manual_resolution["id_emt"] = pd.to_numeric(
    manual_resolution["id_emt"],
    errors="raise",
).astype("Int64")

manual_resolution["pk_municipal"] = pd.to_numeric(
    manual_resolution["pk_municipal"],
    errors="raise",
).astype("Int64")


# ---------------------------------------------------------------
# Validar el origen de las decisiones manuales
# ---------------------------------------------------------------

# Toda decisión manual debe corresponder a un par real
# de identificadores presente en la matriz completa.
pair_matrix_keys = (
    pair_matrix[
        [
            "id_emt",
            "pk_municipal",
        ]
    ]
    .drop_duplicates()
)

manual_pair_existence_check = (
    manual_resolution
    .merge(
        pair_matrix_keys,
        on=[
            "id_emt",
            "pk_municipal",
        ],
        how="left",
        indicator=True,
    )
)

missing_from_pair_matrix = (
    manual_pair_existence_check.loc[
        manual_pair_existence_check[
            "_merge"
        ].ne("both"),
        [
            "id_emt",
            "pk_municipal",
        ],
    ]
)

assert missing_from_pair_matrix.empty, (
    "Existen decisiones manuales cuyos identificadores "
    "no aparecen en la matriz completa de pares:\n"
    + missing_from_pair_matrix.to_string(index=False)
)


# ---------------------------------------------------------------
# Comprobar qué decisiones proceden de la preselección inicial
# ---------------------------------------------------------------

candidate_keys = (
    candidate_pairs[
        [
            "id_emt",
            "pk_municipal",
        ]
    ]
    .drop_duplicates()
)

manual_candidate_check = (
    manual_resolution
    .merge(
        candidate_keys,
        on=[
            "id_emt",
            "pk_municipal",
        ],
        how="left",
        indicator=True,
    )
)

manual_pairs_outside_candidates = (
    manual_candidate_check.loc[
        manual_candidate_check[
            "_merge"
        ].ne("both"),
        [
            "id_emt",
            "pk_municipal",
        ],
    ]
    .copy()
)

observed_manual_exceptions = set(
    map(
        tuple,
        manual_pairs_outside_candidates[
            [
                "id_emt",
                "pk_municipal",
            ]
        ]
        .astype(int)
        .to_numpy(),
    )
)


# Excepciones incorporadas después de la auditoría residual.
# No se amplían las reglas generales de candidatos para evitar
# introducir falsos positivos en todo el conjunto.
EXPECTED_MANUAL_EXCEPTIONS = {
    (45, 7393089),
}

assert (
    observed_manual_exceptions
    == EXPECTED_MANUAL_EXCEPTIONS
), (
    "Las decisiones manuales fuera de candidate_pairs "
    "no coinciden con las excepciones documentadas.\n"
    f"Observadas: {observed_manual_exceptions}\n"
    f"Esperadas: {EXPECTED_MANUAL_EXCEPTIONS}"
)


manual_origin_checks = pd.DataFrame(
    [
        (
            "decisiones_manuales_totales",
            len(manual_resolution),
        ),
        (
            "decisiones_desde_candidate_pairs",
            (
                len(manual_resolution)
                - len(manual_pairs_outside_candidates)
            ),
        ),
        (
            "excepciones_post_auditoria_residual",
            len(manual_pairs_outside_candidates),
        ),
    ],
    columns=[
        "check",
        "valor",
    ],
)

display(
    manual_origin_checks
)

if not manual_pairs_outside_candidates.empty:
    print(
        "Decisiones manuales incorporadas mediante "
        "auditoría residual explícita:"
    )

    display(
        manual_pairs_outside_candidates
    )

# ---------------------------------------------------------------
# Prioridad genérica de las decisiones manuales
# ---------------------------------------------------------------

manual_ids = set(
    manual_resolution["id_emt"]
    .astype(int)
)

manual_pks = set(
    manual_resolution["pk_municipal"]
    .astype(int)
)

automatic_overrides = (
    automatic_resolution.loc[
        automatic_resolution["id_emt"].isin(
            manual_ids
        )
        |
        automatic_resolution["pk_municipal"].isin(
            manual_pks
        )
    ]
    .copy()
)

# Las decisiones manuales prevalecen sin depender
# de una pareja conflictiva codificada de antemano.
automatic_resolution = (
    automatic_resolution.loc[
        ~automatic_resolution["id_emt"].isin(
            manual_ids
        )
        &
        ~automatic_resolution["pk_municipal"].isin(
            manual_pks
        )
    ]
    .copy()
)


# ---------------------------------------------------------------
# Unión final de correspondencias
# ---------------------------------------------------------------

resolved_matches = (
    pd.concat(
        [
            automatic_resolution,
            manual_resolution,
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "pk_municipal",
            "id_emt",
        ]
    )
    .reset_index(drop=True)
)

duplicate_resolved_ids = (
    resolved_matches.loc[
        resolved_matches["id_emt"].duplicated(
            keep=False
        )
    ]
)

duplicate_resolved_pks = (
    resolved_matches.loc[
        resolved_matches[
            "pk_municipal"
        ].duplicated(
            keep=False
        )
    ]
)

assert duplicate_resolved_ids.empty, (
    "Un id_emt aparece en varias correspondencias:\n"
    + duplicate_resolved_ids.to_string(index=False)
)

assert duplicate_resolved_pks.empty, (
    "Un pk_municipal aparece en varias correspondencias:\n"
    + duplicate_resolved_pks.to_string(index=False)
)

assert len(automatic_resolution) == 44
assert len(manual_resolution) == 17
assert len(resolved_matches) == 61

assert 51 not in set(
    resolved_matches["id_emt"]
)

assert 95 not in set(
    resolved_matches["id_emt"]
)

assert 104 not in set(
    resolved_matches["id_emt"]
)

assert 105 not in set(
    resolved_matches["id_emt"]
)

assert (
    (
        resolved_matches["id_emt"].eq(58)
        &
        resolved_matches["pk_municipal"].eq(13468)
    ).sum()
    == 1
)


# ---------------------------------------------------------------
# Identificadores EMT adicionales de entidades consolidadas
# ---------------------------------------------------------------

alias_specs = [{'id_emt_alias': 95,
  'id_emt_referencia': 2,
  'motivo_alias': 'Misma denominación y dirección de Colón; 44,7 m entre registros. El '
                  'significado operativo de las cuatro plazas no puede confirmarse.'},
 {'id_emt_alias': 104,
  'id_emt_referencia': 58,
  'motivo_alias': 'Velázquez-Juan Bravo: dirección ausente, capacidad no informada y '
                  'coordenadas anómalas. Se conserva junto al ID EMT 58.'},
 {'id_emt_alias': 105,
  'id_emt_referencia': 17,
  'motivo_alias': 'Arquitecto Ribera: dirección ausente, capacidad no informada y coordenadas '
                  'anómalas. Se conserva junto al ID EMT 17.'},
 {'id_emt_alias': 69,
  'id_emt_referencia': 17,
  'motivo_alias': 'Barceló y Arquitecto Ribera se sitúan a 18,3 m. Las capacidades se '
                  'conservan sin sumarse porque pueden representar componentes operativos '
                  'distintos.'}]

emt_aliases = pd.DataFrame(
    alias_specs
)

for column in [
    "id_emt_alias",
    "id_emt_referencia",
]:
    emt_aliases[column] = pd.to_numeric(
        emt_aliases[column],
        errors="raise",
    ).astype("Int64")

assert emt_aliases["id_emt_alias"].is_unique

assert not (
    set(emt_aliases["id_emt_alias"])
    & set(resolved_matches["id_emt"])
)

assert set(
    emt_aliases["id_emt_referencia"]
).issubset(
    set(resolved_matches["id_emt"])
)

expected_alias_pairs = {
    (95, 2),
    (104, 58),
    (105, 17),
    (69, 17),
}

observed_alias_pairs = set(
    map(
        tuple,
        emt_aliases[
            [
                "id_emt_alias",
                "id_emt_referencia",
            ]
        ]
        .astype(int)
        .to_numpy(),
    )
)

assert observed_alias_pairs == expected_alias_pairs


# ---------------------------------------------------------------
# Comprobar cobertura antes de construir el inventario
# ---------------------------------------------------------------

resolved_emt_ids = set(
    resolved_matches["id_emt"]
    .astype(int)
)

alias_emt_ids = set(
    emt_aliases["id_emt_alias"]
    .astype(int)
)

all_emt_ids = set(
    emt_clean["id_emt"]
    .astype(int)
)

emt_only_ids_preview = (
    all_emt_ids
    - resolved_emt_ids
    - alias_emt_ids
)

resolved_pks = set(
    resolved_matches["pk_municipal"]
    .astype(int)
)

all_municipal_pks = set(
    municipal_clean["pk_municipal"]
    .astype(int)
)

municipal_only_pks_preview = (
    all_municipal_pks
    - resolved_pks
)

assert len(emt_only_ids_preview) == 22
assert len(municipal_only_pks_preview) == 2

assert (
    resolved_emt_ids
    | alias_emt_ids
    | emt_only_ids_preview
) == all_emt_ids

assert (
    resolved_pks
    | municipal_only_pks_preview
) == all_municipal_pks

n_matches = len(resolved_matches)
n_aliases = len(emt_aliases)
n_solo_emt = len(emt_only_ids_preview)
n_solo_municipal = len(municipal_only_pks_preview)
n_entidades_esperadas = (
    n_matches
    + n_solo_emt
    + n_solo_municipal
)

assert n_matches == 61
assert n_aliases == 4
assert n_solo_emt == 22
assert n_solo_municipal == 2
assert n_entidades_esperadas == 85


resolution_checks = pd.DataFrame(
    [
        (
            "matches_automaticos",
            len(automatic_resolution),
        ),
        (
            "matches_manuales",
            len(manual_resolution),
        ),
        (
            "matches_totales",
            len(resolved_matches),
        ),
        (
            "automaticos_sustituidos_por_manual",
            len(automatic_overrides),
        ),
        (
            "identificadores_emt_alias",
            len(emt_aliases),
        ),
        (
            "entidades_solo_emt_esperadas",
            len(emt_only_ids_preview),
        ),
        (
            "entidades_solo_municipal_esperadas",
            len(municipal_only_pks_preview),
        ),
        (
            "entidades_finales_esperadas",
            n_entidades_esperadas,
        ),
    ],
    columns=[
        "check",
        "valor",
    ],
)

display(
    resolution_checks
)

,check,valor
0,decisiones_manuales_totales,17
1,decisiones_desde_candidate_pairs,16
2,excepciones_post_auditoria_residual,1


Decisiones manuales incorporadas mediante auditoría residual explícita:


,id_emt,pk_municipal
16,45,7393089


,check,valor
0,matches_automaticos,44
1,matches_manuales,17
2,matches_totales,61
3,automaticos_sustituidos_por_manual,0
4,identificadores_emt_alias,4
5,entidades_solo_emt_esperadas,22
6,entidades_solo_municipal_esperadas,2
7,entidades_finales_esperadas,85


### 5.6. Construcción de `inventario_global_emt`

El inventario utiliza una fila por entidad física.

Para las entidades presentes en la fuente municipal se adopta un `parking_uid` basado en `pk_municipal`, ya que este identificador proporciona localización validada, clasificación de acceso e información territorial. Los aparcamientos exclusivos de EMT reciben un identificador basado en `id_emt`.

Las capacidades de ambas fuentes se conservan por separado. No se calcula una capacidad única porque los campos pueden representar plazas públicas, residenciales, totales o subconjuntos operativos distintos.

En las entidades con varios identificadores EMT, `plazas_standard_emt` y `plazas_pmr_emt` corresponden exclusivamente al `id_emt_referencia`. Las capacidades de los identificadores adicionales permanecen disponibles en `emt_clean` y pueden recuperarse mediante `parking_id_map`, pero no se agregan ni sustituyen automáticamente.

La relación técnica `parking_id_map` conserva los 87 identificadores EMT y los 63 identificadores municipales. Colón y Velázquez–Juan Bravo mantienen dos IDs EMT; Arquitecto Ribera conserva tres IDs EMT al incorporar los registros 17, 69 y 105.

El campo `es_parking_emt_referencia` describe únicamente el identificador principal de la entidad. Para evitar interpretaciones incompletas, `es_parking_emt_any` indica si al menos uno de los IDs EMT asociados está marcado como aparcamiento EMT. El output analítico principal continúa siendo `inventario_global_emt`.

In [36]:
# ===============================================================
# CONSTRUCCIÓN DEL INVENTARIO GLOBAL
# ===============================================================

INVENTORY_OUTPUT_PATH = (
    ROOT
    / "data/processed/core/emt"
    / "inventario_global_emt.parquet"
)

ID_MAP_OUTPUT_PATH = (
    ROOT
    / "data/processed/core/emt"
    / "parking_id_map.parquet"
)


# ---------------------------------------------------------------
# Contratos mínimos de entrada
# ---------------------------------------------------------------

required_emt_inventory_columns = {
    "id_emt",
    "nombre_original",
    "nombre",
    "direccion",
    "latitud",
    "longitud",
    "es_parking_emt",
    "plazas_standard",
    "plazas_pmr",
}

required_municipal_inventory_columns = {
    "pk_municipal",
    "nombre_original",
    "nombre",
    "tipo_acceso",
    "direccion",
    "latitud",
    "longitud",
    "cod_distrito",
    "distrito",
    "num_barrio",
    "cod_barrio",
    "barrio",
    "accesibilidad",
    "url_ficha",
    "plazas_automoviles_desc",
    "plazas_publicas_desc",
    "plazas_residentes_desc",
    "plazas_pmr_desc",
    "plazas_electricas_desc",
    "plazas_motocicletas_desc",
}

missing_emt_inventory_columns = sorted(
    required_emt_inventory_columns
    - set(emt_clean.columns)
)

missing_municipal_inventory_columns = sorted(
    required_municipal_inventory_columns
    - set(municipal_clean.columns)
)

assert not missing_emt_inventory_columns, (
    "Faltan columnas EMT necesarias para construir el inventario: "
    f"{missing_emt_inventory_columns}"
)

assert not missing_municipal_inventory_columns, (
    "Faltan columnas municipales necesarias para construir el inventario: "
    f"{missing_municipal_inventory_columns}"
)


# ---------------------------------------------------------------
# Preparar fuentes con nombres inequívocos
# ---------------------------------------------------------------

emt_inventory_source = (
    emt_clean.rename(
        columns={
            "nombre_original": "nombre_original_emt",
            "nombre": "nombre_emt",
            "direccion": "direccion_emt",
            "latitud": "latitud_emt",
            "longitud": "longitud_emt",
            "plazas_standard": "plazas_standard_emt",
            "plazas_pmr": "plazas_pmr_emt",
        }
    )
    .copy()
)

municipal_inventory_source = (
    municipal_clean.rename(
        columns={
            "nombre_original": "nombre_original_municipal",
            "nombre": "nombre_municipal",
            "direccion": "direccion_municipal",
            "latitud": "latitud_municipal",
            "longitud": "longitud_municipal",
            "plazas_automoviles_desc":
                "plazas_automoviles_municipal",
            "plazas_publicas_desc":
                "plazas_publicas_municipal",
            "plazas_residentes_desc":
                "plazas_residentes_municipal",
            "plazas_pmr_desc":
                "plazas_pmr_municipal",
            "plazas_electricas_desc":
                "plazas_electricas_municipal",
            "plazas_motocicletas_desc":
                "plazas_motocicletas_municipal",
        }
    )
    .copy()
)


# ---------------------------------------------------------------
# Parking UID de las correspondencias
# ---------------------------------------------------------------

resolved_matches = resolved_matches.copy()

resolved_matches["parking_uid"] = (
    "MUN_"
    + resolved_matches[
        "pk_municipal"
    ].astype("string")
)

assert resolved_matches["parking_uid"].is_unique

primary_uid_by_emt = (
    resolved_matches
    .set_index("id_emt")[
        "parking_uid"
    ]
    .to_dict()
)


# ---------------------------------------------------------------
# Tabla técnica de identificadores
# ---------------------------------------------------------------

primary_ids_with_alias = set(
    emt_aliases[
        "id_emt_referencia"
    ].astype(int)
)

primary_emt_map = pd.DataFrame(
    {
        "parking_uid":
            resolved_matches["parking_uid"],
        "fuente": "emt",
        "source_id":
            resolved_matches["id_emt"],
        "rol_id": [
            (
                "principal"
                if int(identifier)
                in primary_ids_with_alias
                else "unico"
            )
            for identifier in
            resolved_matches["id_emt"]
        ],
        "motivo":
            resolved_matches["motivo_match"],
    }
)

alias_map = emt_aliases.copy()

alias_map["parking_uid"] = (
    alias_map[
        "id_emt_referencia"
    ]
    .map(primary_uid_by_emt)
)

assert alias_map["parking_uid"].notna().all(), (
    "Algún alias EMT no puede asociarse a un parking_uid principal."
)

alias_emt_map = pd.DataFrame(
    {
        "parking_uid":
            alias_map["parking_uid"],
        "fuente": "emt",
        "source_id":
            alias_map["id_emt_alias"],
        "rol_id": "alias",
        "motivo":
            alias_map["motivo_alias"],
    }
)

matched_primary_ids = set(
    resolved_matches["id_emt"].astype(int)
)

alias_ids = set(
    emt_aliases["id_emt_alias"].astype(int)
)

emt_only_ids = sorted(
    set(emt_clean["id_emt"].astype(int))
    - matched_primary_ids
    - alias_ids
)

emt_only_source = (
    emt_clean.loc[
        emt_clean[
            "id_emt"
        ].astype(int).isin(emt_only_ids)
    ]
    .copy()
)

emt_only_map = pd.DataFrame(
    {
        "parking_uid":
            "EMT_"
            + emt_only_source[
                "id_emt"
            ].astype("string"),
        "fuente": "emt",
        "source_id":
            emt_only_source["id_emt"],
        "rol_id": "unico",
        "motivo":
            "Registro exclusivo de la fuente EMT.",
    }
)

municipal_id_map = pd.DataFrame(
    {
        "parking_uid":
            "MUN_"
            + municipal_clean[
                "pk_municipal"
            ].astype("string"),
        "fuente": "municipal",
        "source_id":
            municipal_clean["pk_municipal"],
        "rol_id": "unico",
        "motivo":
            "Identificador original de la fuente municipal.",
    }
)

parking_id_map = (
    pd.concat(
        [
            primary_emt_map,
            alias_emt_map,
            emt_only_map,
            municipal_id_map,
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "parking_uid",
            "fuente",
            "source_id",
        ]
    )
    .reset_index(drop=True)
)

parking_id_map["source_id"] = pd.to_numeric(
    parking_id_map["source_id"],
    errors="raise",
).astype("Int64")

for column in [
    "parking_uid",
    "fuente",
    "rol_id",
    "motivo",
]:
    parking_id_map[column] = (
        parking_id_map[column]
        .astype("string")
    )

assert not parking_id_map.duplicated(
    subset=[
        "fuente",
        "source_id",
    ]
).any(), (
    "Un identificador de origen aparece asociado a más de una entidad."
)


# ---------------------------------------------------------------
# Entidades presentes en ambas fuentes
# ---------------------------------------------------------------

matched_source = (
    resolved_matches
    .merge(
        emt_inventory_source,
        on="id_emt",
        how="left",
        validate="one_to_one",
    )
    .merge(
        municipal_inventory_source,
        on="pk_municipal",
        how="left",
        validate="one_to_one",
    )
)

assert len(matched_source) == len(resolved_matches)
assert matched_source["nombre_emt"].notna().all()
assert matched_source["nombre_municipal"].notna().all()

matched_inventory = pd.DataFrame(
    {
        "parking_uid":
            matched_source["parking_uid"],
        "estado_integracion": "ambas",
        "metodo_match":
            matched_source["metodo_match"],
        "id_emt_referencia":
            matched_source["id_emt"],
        "pk_municipal":
            matched_source["pk_municipal"],
        "nombre":
            matched_source["nombre_municipal"],
        "direccion":
            matched_source["direccion_municipal"],
        "latitud":
            matched_source["latitud_municipal"],
        "longitud":
            matched_source["longitud_municipal"],
        "fuente_referencia": "municipal",
        "tipo_acceso":
            matched_source["tipo_acceso"],
        "es_parking_emt_referencia":
            matched_source["es_parking_emt"],
        "cod_distrito":
            matched_source["cod_distrito"],
        "distrito":
            matched_source["distrito"],
        "num_barrio":
            matched_source["num_barrio"],
        "cod_barrio":
            matched_source["cod_barrio"],
        "barrio":
            matched_source["barrio"],
        "accesibilidad":
            matched_source["accesibilidad"],
        "url_ficha":
            matched_source["url_ficha"],
        "plazas_standard_emt":
            matched_source["plazas_standard_emt"],
        "plazas_pmr_emt":
            matched_source["plazas_pmr_emt"],
        "plazas_automoviles_municipal":
            matched_source[
                "plazas_automoviles_municipal"
            ],
        "plazas_publicas_municipal":
            matched_source[
                "plazas_publicas_municipal"
            ],
        "plazas_residentes_municipal":
            matched_source[
                "plazas_residentes_municipal"
            ],
        "plazas_pmr_municipal":
            matched_source[
                "plazas_pmr_municipal"
            ],
        "plazas_electricas_municipal":
            matched_source[
                "plazas_electricas_municipal"
            ],
        "plazas_motocicletas_municipal":
            matched_source[
                "plazas_motocicletas_municipal"
            ],
    }
)


# ---------------------------------------------------------------
# Entidades exclusivamente municipales
# ---------------------------------------------------------------

matched_pks = set(
    resolved_matches["pk_municipal"].astype(int)
)

municipal_only_source = (
    municipal_inventory_source.loc[
        ~municipal_inventory_source[
            "pk_municipal"
        ].astype(int).isin(matched_pks)
    ]
    .copy()
)

municipal_only_inventory = pd.DataFrame(
    {
        "parking_uid":
            "MUN_"
            + municipal_only_source[
                "pk_municipal"
            ].astype("string"),
        "estado_integracion": "solo_municipal",
        "metodo_match": pd.NA,
        "id_emt_referencia": pd.NA,
        "pk_municipal":
            municipal_only_source["pk_municipal"],
        "nombre":
            municipal_only_source[
                "nombre_municipal"
            ],
        "direccion":
            municipal_only_source[
                "direccion_municipal"
            ],
        "latitud":
            municipal_only_source[
                "latitud_municipal"
            ],
        "longitud":
            municipal_only_source[
                "longitud_municipal"
            ],
        "fuente_referencia": "municipal",
        "tipo_acceso":
            municipal_only_source["tipo_acceso"],
        "es_parking_emt_referencia": pd.NA,
        "cod_distrito":
            municipal_only_source["cod_distrito"],
        "distrito":
            municipal_only_source["distrito"],
        "num_barrio":
            municipal_only_source["num_barrio"],
        "cod_barrio":
            municipal_only_source["cod_barrio"],
        "barrio":
            municipal_only_source["barrio"],
        "accesibilidad":
            municipal_only_source["accesibilidad"],
        "url_ficha":
            municipal_only_source["url_ficha"],
        "plazas_standard_emt": pd.NA,
        "plazas_pmr_emt": pd.NA,
        "plazas_automoviles_municipal":
            municipal_only_source[
                "plazas_automoviles_municipal"
            ],
        "plazas_publicas_municipal":
            municipal_only_source[
                "plazas_publicas_municipal"
            ],
        "plazas_residentes_municipal":
            municipal_only_source[
                "plazas_residentes_municipal"
            ],
        "plazas_pmr_municipal":
            municipal_only_source[
                "plazas_pmr_municipal"
            ],
        "plazas_electricas_municipal":
            municipal_only_source[
                "plazas_electricas_municipal"
            ],
        "plazas_motocicletas_municipal":
            municipal_only_source[
                "plazas_motocicletas_municipal"
            ],
    }
)


# ---------------------------------------------------------------
# Entidades exclusivamente EMT
# ---------------------------------------------------------------

emt_only_inventory = pd.DataFrame(
    {
        "parking_uid":
            "EMT_"
            + emt_only_source[
                "id_emt"
            ].astype("string"),
        "estado_integracion": "solo_emt",
        "metodo_match": pd.NA,
        "id_emt_referencia":
            emt_only_source["id_emt"],
        "pk_municipal": pd.NA,
        "nombre":
            emt_only_source["nombre"],
        "direccion":
            emt_only_source["direccion"],
        "latitud":
            emt_only_source["latitud"],
        "longitud":
            emt_only_source["longitud"],
        "fuente_referencia": "emt",
        "tipo_acceso": pd.NA,
        "es_parking_emt_referencia":
            emt_only_source["es_parking_emt"],
        "cod_distrito": pd.NA,
        "distrito": pd.NA,
        "num_barrio": pd.NA,
        "cod_barrio": pd.NA,
        "barrio": pd.NA,
        "accesibilidad": pd.NA,
        "url_ficha": pd.NA,
        "plazas_standard_emt":
            emt_only_source["plazas_standard"],
        "plazas_pmr_emt":
            emt_only_source["plazas_pmr"],
        "plazas_automoviles_municipal": pd.NA,
        "plazas_publicas_municipal": pd.NA,
        "plazas_residentes_municipal": pd.NA,
        "plazas_pmr_municipal": pd.NA,
        "plazas_electricas_municipal": pd.NA,
        "plazas_motocicletas_municipal": pd.NA,
    }
)


# ---------------------------------------------------------------
# Inventario global
# ---------------------------------------------------------------

inventario_global_emt = (
    pd.concat(
        [
            matched_inventory,
            municipal_only_inventory,
            emt_only_inventory,
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "estado_integracion",
            "nombre",
            "parking_uid",
        ]
    )
    .reset_index(drop=True)
)

integer_columns = [
    "id_emt_referencia",
    "pk_municipal",
    "cod_distrito",
    "num_barrio",
    "cod_barrio",
    "accesibilidad",
    "plazas_standard_emt",
    "plazas_pmr_emt",
    "plazas_automoviles_municipal",
    "plazas_publicas_municipal",
    "plazas_residentes_municipal",
    "plazas_pmr_municipal",
    "plazas_electricas_municipal",
    "plazas_motocicletas_municipal",
]

for column in integer_columns:
    inventario_global_emt[column] = pd.to_numeric(
        inventario_global_emt[column],
        errors="coerce",
    ).astype("Int64")


# ---------------------------------------------------------------
# Estado EMT agregado a nivel de entidad
# ---------------------------------------------------------------

emt_flags_by_source_id = (
    emt_clean[
        [
            "id_emt",
            "es_parking_emt",
        ]
    ]
    .rename(
        columns={
            "id_emt": "source_id",
        }
    )
    .copy()
)

emt_flags_by_source_id["source_id"] = pd.to_numeric(
    emt_flags_by_source_id["source_id"],
    errors="raise",
).astype("Int64")

emt_id_map_with_flags = (
    parking_id_map.loc[
        parking_id_map["fuente"].eq("emt"),
        [
            "parking_uid",
            "source_id",
        ],
    ]
    .merge(
        emt_flags_by_source_id,
        on="source_id",
        how="left",
        validate="one_to_one",
    )
)

assert emt_id_map_with_flags[
    "es_parking_emt"
].notna().all(), (
    "Algún identificador EMT del mapa técnico no aparece en emt_clean."
)

emt_entity_flags = (
    emt_id_map_with_flags
    .groupby("parking_uid")[
        "es_parking_emt"
    ]
    .any()
)

inventario_global_emt[
    "es_parking_emt_any"
] = (
    inventario_global_emt[
        "parking_uid"
    ]
    .map(emt_entity_flags)
)

for column in [
    "es_parking_emt_referencia",
    "es_parking_emt_any",
]:
    inventario_global_emt[column] = (
        inventario_global_emt[column]
        .astype("boolean")
    )

string_columns = [
    "parking_uid",
    "estado_integracion",
    "metodo_match",
    "nombre",
    "direccion",
    "fuente_referencia",
    "tipo_acceso",
    "distrito",
    "barrio",
    "url_ficha",
]

for column in string_columns:
    inventario_global_emt[column] = (
        inventario_global_emt[column]
        .astype("string")
    )

emt_ids_per_parking = (
    parking_id_map.loc[
        parking_id_map["fuente"].eq("emt")
    ]
    .groupby("parking_uid")
    .size()
)

inventario_global_emt[
    "n_ids_emt"
] = (
    inventario_global_emt[
        "parking_uid"
    ]
    .map(emt_ids_per_parking)
    .fillna(0)
    .astype("Int64")
)

final_inventory_columns = [
    "parking_uid",
    "estado_integracion",
    "metodo_match",
    "id_emt_referencia",
    "pk_municipal",
    "n_ids_emt",
    "nombre",
    "direccion",
    "latitud",
    "longitud",
    "fuente_referencia",
    "tipo_acceso",
    "es_parking_emt_referencia",
    "es_parking_emt_any",
    "cod_distrito",
    "distrito",
    "num_barrio",
    "cod_barrio",
    "barrio",
    "accesibilidad",
    "url_ficha",
    "plazas_standard_emt",
    "plazas_pmr_emt",
    "plazas_automoviles_municipal",
    "plazas_publicas_municipal",
    "plazas_residentes_municipal",
    "plazas_pmr_municipal",
    "plazas_electricas_municipal",
    "plazas_motocicletas_municipal",
]

missing_final_inventory_columns = sorted(
    set(final_inventory_columns)
    - set(inventario_global_emt.columns)
)

assert not missing_final_inventory_columns, (
    "Faltan columnas finales del inventario: "
    f"{missing_final_inventory_columns}"
)

inventario_global_emt = inventario_global_emt[
    final_inventory_columns
]

assert len(inventario_global_emt) == n_entidades_esperadas
assert inventario_global_emt["parking_uid"].is_unique


## 6. Validaciones finales

Las validaciones comprueban:

- una fila por `parking_uid`;
- conservación de los 87 identificadores EMT y los 63 identificadores municipales;
- unicidad de cada identificador dentro de su fuente;
- 61 entidades presentes en ambas fuentes;
- 22 entidades exclusivas de EMT;
- 2 entidades exclusivas de la fuente municipal;
- 85 entidades físicas en total;
- coordenadas válidas para todos los aparcamientos;
- dos entidades con dos IDs EMT y una entidad con tres;
- coherencia entre `n_ids_emt`, `es_parking_emt_referencia` y `es_parking_emt_any`;
- clasificación de acceso completa cuando existe fuente municipal.

In [37]:
# ===============================================================
# VALIDACIONES FINALES
# ===============================================================

integration_counts = (
    inventario_global_emt[
        "estado_integracion"
    ]
    .value_counts()
    .to_dict()
)

mapped_emt_ids = (
    parking_id_map.loc[
        parking_id_map["fuente"].eq("emt"),
        "source_id",
    ]
)

mapped_municipal_ids = (
    parking_id_map.loc[
        parking_id_map["fuente"].eq("municipal"),
        "source_id",
    ]
)

emt_ids_by_uid = (
    parking_id_map.loc[
        parking_id_map["fuente"].eq("emt")
    ]
    .groupby("parking_uid")
    .size()
)

multi_emt_entities = (
    emt_ids_by_uid.loc[
        emt_ids_by_uid.gt(1)
    ]
)

expected_multi_emt_counts = {
    "MUN_13454": 2,
    "MUN_13468": 2,
    "MUN_13470": 3,
}

observed_multi_emt_counts = (
    multi_emt_entities.astype(int).to_dict()
)

assert len(inventario_global_emt) == n_entidades_esperadas

assert inventario_global_emt[
    "parking_uid"
].is_unique

assert inventario_global_emt[
    "parking_uid"
].notna().all()

assert integration_counts == {
    "ambas": n_matches,
    "solo_emt": n_solo_emt,
    "solo_municipal": n_solo_municipal,
}

assert len(mapped_emt_ids) == len(emt_clean)
assert mapped_emt_ids.is_unique

assert set(
    mapped_emt_ids.astype(int)
) == set(
    emt_clean["id_emt"].astype(int)
)

assert len(mapped_municipal_ids) == len(municipal_clean)
assert mapped_municipal_ids.is_unique

assert set(
    mapped_municipal_ids.astype(int)
) == set(
    municipal_clean["pk_municipal"].astype(int)
)

assert not parking_id_map.duplicated(
    subset=[
        "fuente",
        "source_id",
    ]
).any()

assert set(
    parking_id_map["parking_uid"]
) == set(
    inventario_global_emt["parking_uid"]
)

assert observed_multi_emt_counts == expected_multi_emt_counts

expected_n_ids_series = (
    inventario_global_emt[
        "parking_uid"
    ]
    .map(emt_ids_by_uid)
    .fillna(0)
    .astype("Int64")
)

assert inventario_global_emt[
    "n_ids_emt"
].reset_index(drop=True).equals(
    expected_n_ids_series.reset_index(drop=True)
)

assert inventario_global_emt[
    "latitud"
].notna().all()

assert inventario_global_emt[
    "longitud"
].notna().all()

assert inventario_global_emt[
    "latitud"
].between(40.0, 41.0).all()

assert inventario_global_emt[
    "longitud"
].between(-4.5, -3.0).all()

municipal_coverage_mask = (
    inventario_global_emt[
        "pk_municipal"
    ].notna()
)

emt_coverage_mask = (
    inventario_global_emt[
        "n_ids_emt"
    ].gt(0)
)

assert inventario_global_emt.loc[
    municipal_coverage_mask,
    "tipo_acceso",
].notna().all()

assert inventario_global_emt.loc[
    emt_coverage_mask,
    "es_parking_emt_any",
].notna().all()

assert inventario_global_emt.loc[
    ~emt_coverage_mask,
    "es_parking_emt_any",
].isna().all()

assert inventario_global_emt.loc[
    inventario_global_emt["parking_uid"].isin(
        expected_multi_emt_counts
    ),
    "es_parking_emt_any",
].all()

assert inventario_global_emt.loc[
    inventario_global_emt[
        "estado_integracion"
    ].eq("solo_municipal"),
    "es_parking_emt_referencia",
].isna().all()

final_inventory_checks = pd.DataFrame(
    [
        (
            "entidades_inventario",
            len(inventario_global_emt),
        ),
        (
            "parking_uid_duplicados",
            int(
                inventario_global_emt[
                    "parking_uid"
                ]
                .duplicated()
                .sum()
            ),
        ),
        (
            "entidades_ambas_fuentes",
            integration_counts.get(
                "ambas",
                0,
            ),
        ),
        (
            "entidades_solo_emt",
            integration_counts.get(
                "solo_emt",
                0,
            ),
        ),
        (
            "entidades_solo_municipal",
            integration_counts.get(
                "solo_municipal",
                0,
            ),
        ),
        (
            "identificadores_emt_conservados",
            len(mapped_emt_ids),
        ),
        (
            "identificadores_municipales_conservados",
            len(mapped_municipal_ids),
        ),
        (
            "identificadores_emt_alias",
            n_aliases,
        ),
        (
            "entidades_con_varios_ids_emt",
            len(multi_emt_entities),
        ),
        (
            "coordenadas_nulas",
            int(
                inventario_global_emt[
                    [
                        "latitud",
                        "longitud",
                    ]
                ]
                .isna()
                .any(axis=1)
                .sum()
            ),
        ),
        (
            "tipo_acceso_nulo_con_fuente_municipal",
            int(
                inventario_global_emt.loc[
                    municipal_coverage_mask,
                    "tipo_acceso",
                ]
                .isna()
                .sum()
            ),
        ),
        (
            "es_parking_emt_any_nulo_con_id_emt",
            int(
                inventario_global_emt.loc[
                    emt_coverage_mask,
                    "es_parking_emt_any",
                ]
                .isna()
                .sum()
            ),
        ),
    ],
    columns=[
        "check",
        "valor",
    ],
)

display(
    final_inventory_checks
)

print(
    "Shape:",
    inventario_global_emt.shape,
)

display(
    inventario_global_emt.head()
)


,check,valor
0,entidades_inventario,85
1,parking_uid_duplicados,0
2,entidades_ambas_fuentes,61
3,entidades_solo_emt,22
4,entidades_solo_municipal,2
5,identificadores_emt_conservados,87
6,identificadores_municipales_conservados,63
7,identificadores_emt_alias,4
8,entidades_con_varios_ids_emt,3
9,coordenadas_nulas,0


Shape: (85, 29)


,parking_uid,estado_integracion,metodo_match,id_emt_referencia,pk_municipal,n_ids_emt,nombre,direccion,latitud,longitud,fuente_referencia,tipo_acceso,es_parking_emt_referencia,es_parking_emt_any,cod_distrito,distrito,num_barrio,cod_barrio,barrio,accesibilidad,url_ficha,plazas_standard_emt,plazas_pmr_emt,plazas_automoviles_municipal,plazas_publicas_municipal,plazas_residentes_municipal,plazas_pmr_municipal,plazas_electricas_municipal,plazas_motocicletas_municipal
0,MUN_13452,ambas,nombre_exacto_auto,12,13452,1,Almagro,"Calle Almagro, 24",40.429802,-3.693238,municipal,publico,True,True,7,CHAMBERI,4,704,ALMAGRO,0,http://www.madrid.es/sites/v/index.jsp?vgnextchannel=bfa48ab43d6bb410VgnVCM100000171f5a0aRCRD&vgnextoid=0e167339ae51...,453,12,463,<NA>,17,<NA>,<NA>,<NA>
1,MUN_13470,ambas,revision_multisenal,17,13470,3,Arquitecto Ribera,"Calle Barcelo, 2",40.426274,-3.700374,municipal,mixto,False,True,1,CENTRO,4,104,JUSTICIA,0,http://www.madrid.es/sites/v/index.jsp?vgnextchannel=bfa48ab43d6bb410VgnVCM100000171f5a0aRCRD&vgnextoid=a50e15cbed51...,317,7,<NA>,318,298,<NA>,<NA>,<NA>
2,MUN_7393089,ambas,anomalia_espacial_revisada,45,7393089,1,Auditorio Nacional de Música (Príncipe de Vergara),"Calle Cartagena, 178",40.444295,-3.677697,municipal,publico,False,False,5,CHAMARTIN,3,503,CIUDAD JARDIN,1,http://www.madrid.es/sites/v/index.jsp?vgnextchannel=bfa48ab43d6bb410VgnVCM100000171f5a0aRCRD&vgnextoid=ed88f25e040c...,376,0,376,<NA>,<NA>,<NA>,<NA>,<NA>
3,MUN_59060,ambas,revision_multisenal,16,59060,1,Avenida de América (intercambiador),"Avenida America, 9",40.438178,-3.676834,municipal,mixto,False,False,5,CHAMARTIN,2,502,PROSPERIDAD,0,http://www.madrid.es/sites/v/index.jsp?vgnextchannel=bfa48ab43d6bb410VgnVCM100000171f5a0aRCRD&vgnextoid=3f3bc2c6e051...,269,0,<NA>,261,395,<NA>,<NA>,<NA>
4,MUN_5898885,ambas,revision_exacta,7,5898885,1,Avenida de Portugal,"Avenida Portugal, 155",40.411239,-3.738088,municipal,mixto,True,True,10,LATINA,2,1002,PUERTA DEL ANGEL,0,http://www.madrid.es/sites/v/index.jsp?vgnextchannel=bfa48ab43d6bb410VgnVCM100000171f5a0aRCRD&vgnextoid=c313e26f2b3f...,435,14,<NA>,428,445,<NA>,<NA>,<NA>


In [38]:
# ===============================================================
# ESCRITURA CONTROLADA DE OUTPUTS
# ===============================================================

if WRITE_OUTPUTS:
    INVENTORY_OUTPUT_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    inventario_global_emt.to_parquet(
        INVENTORY_OUTPUT_PATH,
        index=False,
    )

    parking_id_map.to_parquet(
        ID_MAP_OUTPUT_PATH,
        index=False,
    )

    written_inventory = pd.read_parquet(
        INVENTORY_OUTPUT_PATH
    )

    written_id_map = pd.read_parquet(
        ID_MAP_OUTPUT_PATH
    )

    assert list(
        written_inventory.columns
    ) == final_inventory_columns

    assert len(
        written_inventory
    ) == n_entidades_esperadas

    assert written_inventory[
        "parking_uid"
    ].is_unique

    assert len(
        written_id_map.loc[
            written_id_map["fuente"].eq("emt")
        ]
    ) == len(emt_clean)

    assert len(
        written_id_map.loc[
            written_id_map["fuente"].eq("municipal")
        ]
    ) == len(municipal_clean)

    assert set(
        written_id_map.loc[
            written_id_map["fuente"].eq("emt"),
            "source_id",
        ].astype(int)
    ) == set(
        emt_clean["id_emt"].astype(int)
    )

    assert set(
        written_id_map.loc[
            written_id_map["fuente"].eq("municipal"),
            "source_id",
        ].astype(int)
    ) == set(
        municipal_clean["pk_municipal"].astype(int)
    )

    print(
        "Inventario escrito:",
        INVENTORY_OUTPUT_PATH,
    )

    print(
        "Mapa de identificadores escrito:",
        ID_MAP_OUTPUT_PATH,
    )

else:
    print(
        "Escritura desactivada. "
        "Revisar los checks finales antes de cambiar "
        "WRITE_OUTPUTS=True."
    )

Inventario escrito: /Users/hugo/TFM_parking_madrid/data/processed/core/emt/inventario_global_emt.parquet
Mapa de identificadores escrito: /Users/hugo/TFM_parking_madrid/data/processed/core/emt/parking_id_map.parquet


## 7. Conclusiones y decisiones habilitadas

La integración genera un inventario provisionalmente cerrado de 85 entidades físicas:

- 61 aparcamientos presentes en ambas fuentes;
- 22 registros exclusivos de EMT;
- 2 registros exclusivos de la fuente municipal: Alcántara y Usera.

Los 87 identificadores EMT y los 63 identificadores municipales se conservan mediante `parking_id_map`. Se consolidan cuatro IDs EMT como alias: Colón (`95`), Velázquez–Juan Bravo (`104`) y Arquitecto Ribera (`105` y `69`).

Las coordenadas municipales se adoptan para las entidades presentes en dicha fuente. De este modo, las anomalías espaciales de los IDs 45, 101, 104 y 105 no se trasladan a la representación cartográfica.

Las capacidades de ambas fuentes permanecen separadas porque no representan necesariamente el mismo concepto. Tampoco se suman las capacidades de los alias.

Las consolidaciones de Colón y Arquitecto Ribera-Barceló, así como las correspondencias manuales de Santo Domingo y el Auditorio Nacional, constituyen decisiones analíticas de resolución de entidades y no equivalencias administrativas confirmadas por las fuentes. No puedo confirmar dichas equivalencias de forma oficial. Deberán reevaluarse si las series posteriores de ocupación muestran identificadores con comportamientos operativos independientes.

El inventario queda preparado para:

- su representación en el mapa integrado SER–EMT;
- los posteriores joins con ocupación histórica, mensual y en tiempo real;
- la clasificación de alternativas de estacionamiento fuera de la vía pública;
- el análisis territorial por distrito y barrio cuando existe cobertura municipal.

La ejecución final ha escrito y validado los dos outputs limpios de fuente, el inventario integrado y el mapa técnico de identificadores.